# LeetCode 432 — All O`one Data Structure bằng Python (Hard)

Notebook này giảng nhập môn một bài **thiết kế class** (design) kinh điển: xây một cấu trúc dữ liệu đếm số lần xuất hiện của các chuỗi, và **lúc nào cũng** trả lời ngay được "chuỗi nào đang có số đếm lớn nhất / nhỏ nhất". Skeleton Python3 trên LeetCode:

```python
class AllOne:

    def __init__(self):


    def inc(self, key: str) -> None:


    def dec(self, key: str) -> None:


    def getMaxKey(self) -> str:


    def getMinKey(self) -> str:



# Your AllOne object will be instantiated and called as such:
# obj = AllOne()
# obj.inc(key)
# obj.dec(key)
# param_3 = obj.getMaxKey()
# param_4 = obj.getMinKey()
```

Tên bài đọc là "All O-one": chơi chữ "All O(1)" — **mọi** thao tác đều phải chạy trong `O(1)` trung bình.

Mục tiêu của bài học:

1. Đọc được input dạng chuỗi lệnh `["AllOne","inc",...]`, `[[],["hello"],...]` và tự viết hàm `run_commands` để test giống hệt LeetCode.
2. Đi từ cách người mới tự nghĩ ra (**dict đếm + quét tìm max/min**, `O(N)` mỗi lần hỏi) → **dict + heap "lazy"** (`O(log q)`, có hạn chế) → **danh sách liên kết đôi các "bucket"** (`O(1)` cho mọi thao tác). Mỗi bước hiểu rõ cách cũ vướng ở đâu.
3. Học **heap / `heapq`** và kỹ thuật **lazy deletion** (xóa lười) từ con số 0.
4. Học **danh sách liên kết** (linked list) từ căn bản: liên kết đơn → liên kết đôi → **sentinel** `head` / `tail`; tự cài chèn / xóa một node trong `O(1)`.
5. Hiểu mấu chốt của bài: `inc` / `dec` chỉ đẩy một key sang **count kề bên** (`c + 1` hoặc `c - 1`), nên bucket mới luôn được chèn **ngay cạnh** bucket cũ.
6. Vẽ ASCII danh sách bucket sau **từng** lệnh, trace kỹ khi nào `inc` / `dec` **tạo** bucket mới và khi nào **xóa** bucket rỗng.
7. Nắm các invariant (bất biến) để giải thích vì sao thuật toán đúng; tự kiểm chứng bằng test chéo ngẫu nhiên; tránh các lỗi kinh điển khi thao tác con trỏ `prev` / `next`.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Hãy thiết kế một cấu trúc dữ liệu lưu **số đếm** (count) của các chuỗi, có khả năng trả về chuỗi có số đếm nhỏ nhất và lớn nhất.
>
> Cài đặt class `AllOne`:
>
> - `AllOne()` khởi tạo object của cấu trúc dữ liệu.
> - `inc(String key)` tăng số đếm của chuỗi `key` thêm `1`. Nếu `key` chưa có trong cấu trúc dữ liệu thì chèn nó vào với số đếm `1`.
> - `dec(String key)` giảm số đếm của chuỗi `key` đi `1`. Nếu sau khi giảm, số đếm của `key` bằng `0` thì xóa `key` khỏi cấu trúc dữ liệu. Đề **đảm bảo** `key` có tồn tại trong cấu trúc dữ liệu trước khi gọi `dec`.
> - `getMaxKey()` trả về **một** trong các key có số đếm lớn nhất. Nếu không có phần tử nào, trả về chuỗi rỗng `""`.
> - `getMinKey()` trả về **một** trong các key có số đếm nhỏ nhất. Nếu không có phần tử nào, trả về chuỗi rỗng `""`.
>
> **Lưu ý:** mỗi hàm phải chạy trong độ phức tạp thời gian `O(1)` **trung bình**.

**Example 1**

```text
Input
["AllOne", "inc", "inc", "getMaxKey", "getMinKey", "inc", "getMaxKey", "getMinKey"]
[[], ["hello"], ["hello"], [], [], ["leet"], [], []]
Output
[null, null, null, "hello", "hello", null, "hello", "leet"]

Explanation
AllOne allOne = new AllOne();
allOne.inc("hello");
allOne.inc("hello");
allOne.getMaxKey(); // return "hello"
allOne.getMinKey(); // return "hello"
allOne.inc("leet");
allOne.getMaxKey(); // return "hello"
allOne.getMinKey(); // return "leet"
```

**Constraints**

- `1 <= key.length <= 10`
- `key` chỉ gồm chữ cái tiếng Anh viết thường.
- Đề đảm bảo với mỗi lời gọi `dec`, `key` đang tồn tại trong cấu trúc dữ liệu.
- Có tối đa `5 * 10^4` lời gọi `inc`, `dec`, `getMaxKey` và `getMinKey` (tính chung).

### Ký hiệu dùng suốt notebook

- `N` = số key **khác nhau** đang có mặt trong cấu trúc dữ liệu (count ≥ 1) tại thời điểm đang xét.
- `q` = tổng số lời gọi hàm (theo đề `q <= 5·10^4`).
- `cnt[key]` = số đếm hiện tại của `key`.

### 1.1 Đọc input / output của bài "thiết kế class"

Đây **không** phải dạng "cho mảng, trả kết quả". LeetCode mô tả một **kịch bản gọi hàm** bằng 2 list song song: list thứ nhất là **tên lệnh**, list thứ hai là **tham số** của lệnh tương ứng. Ta lần theo Example 1 và ghi lại số đếm sau mỗi lệnh:

| Vị trí | Lệnh | Tham số | Số đếm sau lệnh | Output | Giải thích |
|---:|---|---|---|---|---|
| 0 | `"AllOne"` | `[]` | `{}` | `null` | tạo object rỗng |
| 1 | `"inc"` | `["hello"]` | `{hello: 1}` | `null` | key mới → count 1 |
| 2 | `"inc"` | `["hello"]` | `{hello: 2}` | `null` | 1 → 2 |
| 3 | `"getMaxKey"` | `[]` | `{hello: 2}` | `"hello"` | chỉ có 1 key |
| 4 | `"getMinKey"` | `[]` | `{hello: 2}` | `"hello"` | chỉ có 1 key |
| 5 | `"inc"` | `["leet"]` | `{hello: 2, leet: 1}` | `null` | key mới → count 1 |
| 6 | `"getMaxKey"` | `[]` | như trên | `"hello"` | 2 > 1 |
| 7 | `"getMinKey"` | `[]` | như trên | `"leet"` | 1 < 2 |

- `null` là cách Java / JSON viết "không có giá trị"; trong Python tương ứng là `None`. Hàm `inc` / `dec` có kiểu trả về `None` nên output là `null`.
- Chú ý chữ **"một trong các key"**: nếu `a` và `b` cùng có count lớn nhất thì trả `"a"` hay `"b"` đều được chấp nhận. Khi tự test, ta **không** được so output với một chuỗi cố định trong trường hợp hòa; phải kiểm tra "count của key trả về có đúng là lớn nhất không".

### 1.2 Vì sao bài này là Hard? "O(1) trung bình" nghĩa là gì?

Nếu chỉ cần **đúng** thì bài rất dễ: một dict đếm, khi hỏi max / min thì quét hết dict. Cái khó nằm ở dòng **"mỗi hàm phải chạy trong `O(1)` trung bình"**:

- `O(1)` nghĩa là thời gian mỗi lời gọi **không tăng theo** số key `N`. Có 10 key hay 10 000 key thì `getMaxKey` vẫn tốn khoảng cùng một số bước.
- Chữ **"trung bình"** (average) có mặt vì ta được phép dùng **hash table** (`dict`, `set` của Python): tra cứu / thêm / xóa tốn `O(1)` trung bình (trong trường hợp xấu hiếm gặp có thể chậm hơn do va chạm hash).

Như vậy ta cần một cấu trúc vừa cập nhật count nhanh, vừa luôn "biết sẵn" đâu là count lớn nhất / nhỏ nhất — **kể cả** khi key đang giữ max/min bị giảm, bị xóa. Đó là chỗ phải sáng tạo: kết hợp **dict** với **danh sách liên kết đôi**.

## 2. Nền tảng Python

### 2.1 `dict` dùng để đếm

`dict` (dictionary — từ điển) lưu các cặp **key → value**. Ở bài này key là chuỗi, value là số đếm.

- `cnt.get(key, 0)`: đọc value của `key`; nếu chưa có thì trả về mặc định `0` (không báo lỗi). Nhờ vậy `cnt[key] = cnt.get(key, 0) + 1` xử lý gọn cả key mới lẫn key cũ.
- `del cnt[key]`: xóa hẳn key khỏi dict (dùng khi count về `0`).
- `key in cnt`: kiểm tra key có trong dict không — `O(1)` trung bình.
- `max(cnt, key=cnt.get)`: duyệt **mọi key** trong dict, so sánh theo `cnt.get(k)` (tức count), trả về key có count lớn nhất. Chú ý: đây là phép **quét**, tốn `O(N)`. Nếu dict rỗng, `max` báo lỗi `ValueError`.

In [1]:
cnt = {}
for word in ["hello", "hello", "leet", "code", "hello", "code"]:
    cnt[word] = cnt.get(word, 0) + 1      # key mới -> get trả 0 -> count 1
print(cnt)                                # {'hello': 3, 'leet': 1, 'code': 2}

print("max:", max(cnt, key=cnt.get))      # hello (count 3)
print("min:", min(cnt, key=cnt.get))      # leet  (count 1)

cnt["leet"] -= 1
if cnt["leet"] == 0:
    del cnt["leet"]                       # count về 0 -> xóa hẳn
print(cnt, "| 'leet' in cnt ?", "leet" in cnt)

try:
    max({}, key={}.get)
except ValueError as e:
    print("max trên dict rỗng -> ValueError:", e)

{'hello': 3, 'leet': 1, 'code': 2}
max: hello
min: leet
{'hello': 3, 'code': 2} | 'leet' in cnt ? False
max trên dict rỗng -> ValueError: max() iterable argument is empty


### 2.2 `set` — tập hợp các key, và mẹo `next(iter(s))`

`set` là một "túi" chứa các phần tử **không trùng nhau**, không có thứ tự. Lời giải chính sẽ gom các key có **cùng count** vào một `set`.

| Thao tác | Cú pháp | Chi phí trung bình |
|---|---|---|
| tạo rỗng | `s = set()` (chú ý: `{}` là **dict** rỗng, không phải set!) | `O(1)` |
| thêm | `s.add(x)` | `O(1)` |
| xóa (bắt buộc có) | `s.remove(x)` — báo `KeyError` nếu không có | `O(1)` |
| xóa (có thì xóa) | `s.discard(x)` — không báo lỗi | `O(1)` |
| kiểm tra | `x in s` | `O(1)` |
| rỗng chưa? | `not s` là `True` khi `s` rỗng | `O(1)` |

**Lấy ra "một phần tử bất kỳ" mà không xóa nó**: `set` không có chỉ số `s[0]`. Ta dùng `next(iter(s))`:

- `iter(s)` tạo một **iterator** — "con trỏ đọc" đi lần lượt qua các phần tử.
- `next(...)` đọc phần tử **đầu tiên** của iterator đó rồi dừng — không duyệt hết set.
- Nếu set rỗng, `next` báo lỗi `StopIteration`. Vì vậy phải chắc set không rỗng trước khi gọi.

Lưu ý thực tế: thứ tự duyệt của `set` chứa chuỗi **có thể khác nhau giữa các lần chạy Python** (Python ngẫu nhiên hóa hàm hash của chuỗi). Khi in set cho dễ đọc, notebook sẽ in `sorted(s)`.

In [2]:
s = set()
s.add("b"); s.add("a"); s.add("a")         # "a" thêm 2 lần vẫn chỉ có 1
print(sorted(s), len(s))                   # ['a', 'b'] 2
print("a" in s, "z" in s)                  # True False

any_key = next(iter(s))                    # lấy 1 phần tử bất kỳ, KHÔNG xóa
print("một key bất kỳ:", any_key in {"a", "b"}, "| set vẫn còn:", sorted(s))

s.discard("z")                             # không có "z" -> không lỗi
s.remove("a"); s.remove("b")
print("rỗng?", not s)

try:
    next(iter(s))
except StopIteration:
    print("next(iter(set rỗng)) -> StopIteration")

print(type({}), type(set()))               # {} là dict, set() mới là set

['a', 'b'] 2
True False
một key bất kỳ: True | set vẫn còn: ['a', 'b']
rỗng? True
next(iter(set rỗng)) -> StopIteration
<class 'dict'> <class 'set'>


### 2.3 Class và trạng thái: object "nhớ" dữ liệu giữa các lần gọi

Bài thiết kế yêu cầu một **class**. Mỗi object tạo từ class có các **thuộc tính** riêng (gắn với `self`), tồn tại suốt đời object. Vì vậy lần gọi `inc` sau vẫn thấy được dữ liệu mà lần gọi trước để lại.

- `__init__(self)` chạy **một lần** khi tạo object: nơi khởi tạo mọi cấu trúc dữ liệu (`self.cnt = {}`).
- Các method khác (`inc`, `dec`, ...) đọc / sửa `self.cnt`.

### 2.4 Tự viết hàm chạy chuỗi lệnh kiểu LeetCode

Hàm `run_commands(cls, ops, args)`:

1. Tạo object: `obj = cls(*args[0])`.
2. Với mỗi cặp (`op`, `a`) còn lại: lấy method theo **tên dạng chuỗi** bằng `getattr(obj, op)` rồi gọi với `*a`.
3. Gom kết quả vào list (lời gọi khởi tạo cho `None`, tức `null`).

Hai công cụ mới:

- `getattr(obj, "inc")` lấy method của object theo tên. `getattr(obj, "inc")("a")` giống hệt `obj.inc("a")`.
- `*a` "rải" (unpack) list thành các đối số: `f(*["a"])` giống `f("a")`, `f(*[])` giống `f()`. Nhờ vậy một dòng code gọi được cả `inc("a")` lẫn `getMaxKey()`.

Hàm `show(outputs)` in kết quả giống LeetCode: `None` → `null`, chuỗi có dấu ngoặc kép (dùng `json.dumps`).

Ta truyền **class thật** vào tham số `cls`, nên cùng một chuỗi lệnh có thể chạy cho nhiều cách cài đặt khác nhau (`AllOneScan`, `AllOneLazyHeap`, `AllOneBuckets`, ...).

In [3]:
import json


class KeyCounter:
    # Class tí hon để thử: chỉ đếm và đọc lại count
    def __init__(self):
        self.cnt = {}                      # trạng thái gắn với object

    def inc(self, key):
        self.cnt[key] = self.cnt.get(key, 0) + 1

    def get(self, key):
        return self.cnt.get(key, 0)


def run_commands(cls, ops, args):
    # Chạy chuỗi lệnh kiểu LeetCode: ops[0] là lệnh tạo object, args[0] là tham số của __init__
    obj = cls(*args[0])
    outputs = [None]                       # lời gọi khởi tạo -> null
    for op, a in zip(ops[1:], args[1:]):
        method = getattr(obj, op)          # lấy method theo tên, ví dụ "inc"
        outputs.append(method(*a))         # gọi với tham số rải từ list a
    return outputs


def show(outputs):
    # In giống LeetCode: None -> null, chuỗi -> "..."
    return "[" + ", ".join("null" if x is None else json.dumps(x) for x in outputs) + "]"


kc = KeyCounter()
getattr(kc, "inc")(*["x"])                 # giống kc.inc("x")
kc.inc("x")
print("count của x:", kc.get("x"))         # 2 -> object nhớ trạng thái

out = run_commands(KeyCounter, ["KeyCounter", "inc", "inc", "get", "get"],
                   [[], ["a"], ["a"], ["a"], ["b"]])
print(show(out))                           # [null, null, null, 2, 0]

EX1_OPS = ["AllOne", "inc", "inc", "getMaxKey", "getMinKey", "inc", "getMaxKey", "getMinKey"]
EX1_ARGS = [[], ["hello"], ["hello"], [], [], ["leet"], [], []]
EX1_OUT = [None, None, None, "hello", "hello", None, "hello", "leet"]
print("Example 1 expected:", show(EX1_OUT))

count của x: 2
[null, null, null, 2, 0]
Example 1 expected: [null, null, null, "hello", "hello", null, "hello", "leet"]


## 3. Cách 1 — Dict đếm + quét tìm max / min

Cách người mới nghĩ ra đầu tiên, và hoàn toàn đúng:

- Giữ một dict `cnt` đếm số lần của mỗi key.
- `inc(key)`: `cnt[key] = cnt.get(key, 0) + 1`.
- `dec(key)`: giảm 1; nếu về `0` thì `del cnt[key]` (đề yêu cầu xóa khỏi cấu trúc).
- `getMaxKey()` / `getMinKey()`: nếu dict rỗng trả `""`; ngược lại quét toàn bộ dict bằng `max` / `min` với `key=cnt.get`.

Đây sẽ là **baseline** (mốc chuẩn) để đối chiếu các cách nhanh hơn: code ngắn, khó sai.

In [4]:
class AllOneScan:
    # Cách 1: dict đếm; hỏi max/min thì quét toàn bộ dict -> O(N)
    def __init__(self):
        self.cnt = {}

    def inc(self, key: str) -> None:
        self.cnt[key] = self.cnt.get(key, 0) + 1

    def dec(self, key: str) -> None:
        self.cnt[key] -= 1
        if self.cnt[key] == 0:
            del self.cnt[key]              # count về 0 -> xóa key

    def getMaxKey(self) -> str:
        if not self.cnt:
            return ""
        return max(self.cnt, key=self.cnt.get)

    def getMinKey(self) -> str:
        if not self.cnt:
            return ""
        return min(self.cnt, key=self.cnt.get)


got = run_commands(AllOneScan, EX1_OPS, EX1_ARGS)
print(show(got))
assert got == EX1_OUT

[null, null, null, "hello", "hello", null, "hello", "leet"]


In [5]:
# Thêm một kịch bản có dec về 0 và hỏi khi rỗng
ops = ["AllOne", "inc", "inc", "inc", "dec", "getMaxKey", "getMinKey", "dec", "dec", "getMaxKey", "getMinKey"]
args = [[], ["a"], ["b"], ["b"], ["a"], [], [], ["b"], ["b"], [], []]
print(show(run_commands(AllOneScan, ops, args)))
# a: 1 -> 0 bị xóa; còn b:2 -> max = min = "b"; rồi b về 0 -> rỗng -> "" và ""

[null, null, null, null, null, "b", "b", null, null, "", ""]


### Độ phức tạp cách 1 và chỗ vướng

| Thao tác | Chi phí |
|---|---|
| `inc`, `dec` | `O(1)` trung bình (vài thao tác dict) |
| `getMaxKey`, `getMinKey` | **`O(N)`** — phải nhìn qua **mọi** key |
| Bộ nhớ | `O(N)` |

Chỗ vướng: mỗi lần hỏi, ta **quên sạch** kết quả lần trước và quét lại từ đầu. Kịch bản xấu theo constraint: `inc` 25 000 key khác nhau, rồi gọi 25 000 lần `getMaxKey` → khoảng `25 000 × 25 000 ≈ 6·10^8` bước so sánh. Quá chậm, và chắc chắn không phải `O(1)`.

Ý tưởng tự nhiên tiếp theo: **giữ sẵn** max/min trong một cấu trúc "luôn biết phần tử lớn nhất" — đó là **heap**.

## 4. Nền tảng: Heap, `heapq` và lazy deletion

### 4.1 Heap là gì?

**Heap** (min-heap) là một list được sắp đặt đặc biệt sao cho **phần tử nhỏ nhất luôn nằm ở `h[0]`**. Nó không sắp xếp toàn bộ list — chỉ đảm bảo "đỉnh" là nhỏ nhất. Đổi lại, thêm / lấy đỉnh chỉ tốn `O(log m)` với `m` là số phần tử trong heap.

Module `heapq` của thư viện chuẩn:

- `heapq.heappush(h, x)`: thêm `x`, `O(log m)`.
- `heapq.heappop(h)`: lấy ra và xóa phần tử nhỏ nhất, `O(log m)`.
- `h[0]`: **xem** phần tử nhỏ nhất mà không xóa, `O(1)`.

Hai mẹo dùng trong bài:

- **Tuple so sánh từng thành phần**: `(1, "b") < (2, "a")` vì `1 < 2`; nếu thành phần đầu bằng nhau thì so tiếp thành phần sau: `(1, "a") < (1, "b")`. Ta đẩy `(count, key)` vào heap để heap sắp theo count.
- **Python chỉ có min-heap**. Muốn **max-heap** thì đẩy số âm: `(-count, key)`. Count lớn nhất → `-count` nhỏ nhất → nằm ở đỉnh.

Còn một nút thắt: heap **không** cho sửa / xóa một phần tử ở giữa trong `O(log m)` (phải tìm nó trước, tốn `O(m)`). Mà ở bài này count của key thay đổi liên tục...

In [6]:
import heapq

h = []
for item in [(3, "c"), (1, "b"), (2, "a"), (1, "a")]:
    heapq.heappush(h, item)
print("mảng heap bên trong:", h)            # không sắp xếp hết, nhưng h[0] nhỏ nhất
print("đỉnh h[0]:", h[0])                   # (1, 'a'): count 1, hòa thì so tiếp key
print("pop:", heapq.heappop(h), heapq.heappop(h), "| còn lại:", h)

# max-heap bằng số âm
mh = []
for c, k in [(3, "c"), (1, "b"), (5, "z")]:
    heapq.heappush(mh, (-c, k))
neg_c, k = mh[0]
print("key có count lớn nhất:", k, "count =", -neg_c)   # z 5

mảng heap bên trong: [(1, 'a'), (1, 'b'), (2, 'a'), (3, 'c')]
đỉnh h[0]: (1, 'a')
pop: (1, 'a') (1, 'b') | còn lại: [(2, 'a'), (3, 'c')]
key có count lớn nhất: z count = 5


### 4.2 Lazy deletion — "xóa lười"

Hình dung heap là một **chồng vé** ghi `(count, key)`. Khi count của key đổi, thay vì lục tìm vé cũ để xé (tốn `O(m)`), ta **cứ để vé cũ nằm đó** và **đẩy thêm một vé mới** ghi count mới.

Lúc cần đọc đỉnh, ta **kiểm tra vé ở đỉnh còn đúng không** bằng dict `cnt` (nguồn sự thật):

- Nếu `cnt.get(key, 0) == count ghi trên vé` → vé còn đúng → dùng được.
- Nếu không → vé đã **lỗi thời** (stale) → `heappop` bỏ đi, xem vé tiếp theo.

Mỗi vé bị bỏ **đúng một lần**, nên tổng công dọn dẹp không vượt quá tổng số vé đã đẩy. Cell dưới thử với min-heap: `a` từng có count 1 rồi tăng lên 3.

In [7]:
cnt = {"a": 3, "b": 2}
minh = [(1, "a"), (2, "a"), (2, "b"), (3, "a")]   # các vé (count, key), có vé cũ của "a"
heapq.heapify(minh)                                # biến list thành heap, O(m)

while minh and cnt.get(minh[0][1], 0) != minh[0][0]:
    print("bỏ vé lỗi thời:", heapq.heappop(minh))
print("đỉnh hợp lệ:", minh[0], "-> getMinKey =", minh[0][1])

bỏ vé lỗi thời: (1, 'a')
bỏ vé lỗi thời: (2, 'a')
đỉnh hợp lệ: (2, 'b') -> getMinKey = b


## 5. Cách 2 — Dict đếm + hai heap lazy

- `cnt`: dict đếm (nguồn sự thật).
- `maxh`: max-heap chứa vé `(-count, key)`; `minh`: min-heap chứa vé `(count, key)`.
- `inc` / `dec`: cập nhật `cnt` như cách 1, rồi đẩy **vé mới** vào cả hai heap (nếu count mới > 0). Không đụng tới vé cũ.
- `getMaxKey`: bỏ các vé lỗi thời ở đỉnh `maxh`; nếu heap rỗng trả `""`, ngược lại trả key ở đỉnh. `getMinKey` tương tự với `minh`.

Một điểm tinh tế: vé cũ `(1, "a")` có thể "sống lại" — nếu `a` đi 1 → 2 → 1 thì vé `(1, "a")` cũ lại khớp `cnt`. Điều đó không sao, vì vé đó nói đúng sự thật hiện tại.

In [8]:
class AllOneLazyHeap:
    # Cách 2: dict đếm + 2 heap chứa vé (count, key); vé lỗi thời bị bỏ khi nổi lên đỉnh
    def __init__(self):
        self.cnt = {}
        self.maxh = []     # (-count, key)
        self.minh = []     # (count, key)

    def _push(self, key):
        c = self.cnt.get(key, 0)
        if c > 0:
            heapq.heappush(self.maxh, (-c, key))
            heapq.heappush(self.minh, (c, key))

    def inc(self, key: str) -> None:
        self.cnt[key] = self.cnt.get(key, 0) + 1
        self._push(key)

    def dec(self, key: str) -> None:
        self.cnt[key] -= 1
        if self.cnt[key] == 0:
            del self.cnt[key]
        self._push(key)

    def getMaxKey(self) -> str:
        h = self.maxh
        while h and self.cnt.get(h[0][1], 0) != -h[0][0]:   # vé lỗi thời -> bỏ
            heapq.heappop(h)
        return h[0][1] if h else ""

    def getMinKey(self) -> str:
        h = self.minh
        while h and self.cnt.get(h[0][1], 0) != h[0][0]:
            heapq.heappop(h)
        return h[0][1] if h else ""


got = run_commands(AllOneLazyHeap, EX1_OPS, EX1_ARGS)
print(show(got))
assert got == EX1_OUT
print(show(run_commands(AllOneLazyHeap, ops, args)))   # cùng kịch bản dec về 0 ở trên

[null, null, null, "hello", "hello", null, "hello", "leet"]
[null, null, null, null, null, "b", "b", null, null, "", ""]


### 5.1 Trace cách 2: nhìn các vé lỗi thời bị bỏ

Chuỗi lệnh: `inc a, inc a, inc b, getMaxKey, getMinKey, dec a, dec a, getMinKey, getMaxKey, inc b, getMaxKey`. (Heap được in theo thứ tự đã sắp xếp cho dễ đọc; bên trong vẫn là mảng heap.)

| # | Lệnh | `cnt` sau lệnh | Chuyện gì xảy ra ở heap |
|---:|---|---|---|
| 1 | `inc a` | `{a:1}` | đẩy `(-1,a)`, `(1,a)` |
| 2 | `inc a` | `{a:2}` | đẩy `(-2,a)`, `(2,a)`; vé `(1,a)` giờ lỗi thời nhưng vẫn nằm đó |
| 3 | `inc b` | `{a:2, b:1}` | đẩy `(-1,b)`, `(1,b)` |
| 4 | `getMaxKey` | | đỉnh `(-2,a)`: `cnt[a]=2` khớp → `"a"`, bỏ 0 vé |
| 5 | `getMinKey` | | đỉnh `(1,a)`: `cnt[a]=2 ≠ 1` → bỏ; đỉnh `(1,b)` khớp → `"b"`, bỏ 1 vé |
| 6 | `dec a` | `{a:1, b:1}` | đẩy `(-1,a)`, `(1,a)` |
| 7 | `dec a` | `{b:1}` | `a` bị xóa, count 0 → không đẩy vé |
| 8 | `getMinKey` | | đỉnh `(1,a)`: `a` không còn → bỏ; `(1,b)` khớp → `"b"`, bỏ 1 vé |
| 9 | `getMaxKey` | | `(-2,a)`, `(-1,a)`, `(-1,a)` đều lỗi thời → bỏ 3 vé; `(-1,b)` → `"b"` |
| 10 | `inc b` | `{b:2}` | đẩy `(-2,b)`, `(2,b)` |
| 11 | `getMaxKey` | | đỉnh `(-2,b)` khớp → `"b"` |

In [9]:
def trace_lazy_heap(ops, args):
    obj = AllOneLazyHeap()
    for i, (op, a) in enumerate(zip(ops[1:], args[1:]), start=1):
        if op in ("inc", "dec"):
            getattr(obj, op)(*a)
            print(f"#{i:<2} {op}({a[0]!r}): cnt={obj.cnt}  maxh={sorted(obj.maxh)}  minh={sorted(obj.minh)}")
        else:
            heap = obj.maxh if op == "getMaxKey" else obj.minh
            before = len(heap)
            res = getattr(obj, op)()
            print(f"#{i:<2} {op}() -> {res!r}  (bỏ {before - len(heap)} vé lỗi thời)")


T_OPS = ["AllOne", "inc", "inc", "inc", "getMaxKey", "getMinKey", "dec", "dec",
         "getMinKey", "getMaxKey", "inc", "getMaxKey"]
T_ARGS = [[], ["a"], ["a"], ["b"], [], [], ["a"], ["a"], [], [], ["b"], []]
trace_lazy_heap(T_OPS, T_ARGS)

#1  inc('a'): cnt={'a': 1}  maxh=[(-1, 'a')]  minh=[(1, 'a')]
#2  inc('a'): cnt={'a': 2}  maxh=[(-2, 'a'), (-1, 'a')]  minh=[(1, 'a'), (2, 'a')]
#3  inc('b'): cnt={'a': 2, 'b': 1}  maxh=[(-2, 'a'), (-1, 'a'), (-1, 'b')]  minh=[(1, 'a'), (1, 'b'), (2, 'a')]
#4  getMaxKey() -> 'a'  (bỏ 0 vé lỗi thời)
#5  getMinKey() -> 'b'  (bỏ 1 vé lỗi thời)
#6  dec('a'): cnt={'a': 1, 'b': 1}  maxh=[(-2, 'a'), (-1, 'a'), (-1, 'a'), (-1, 'b')]  minh=[(1, 'a'), (1, 'b'), (2, 'a')]
#7  dec('a'): cnt={'b': 1}  maxh=[(-2, 'a'), (-1, 'a'), (-1, 'a'), (-1, 'b')]  minh=[(1, 'a'), (1, 'b'), (2, 'a')]
#8  getMinKey() -> 'b'  (bỏ 1 vé lỗi thời)
#9  getMaxKey() -> 'b'  (bỏ 3 vé lỗi thời)
#10 inc('b'): cnt={'b': 2}  maxh=[(-2, 'b'), (-1, 'b')]  minh=[(1, 'b'), (2, 'a'), (2, 'b')]
#11 getMaxKey() -> 'b'  (bỏ 0 vé lỗi thời)


### 5.2 Độ phức tạp cách 2 và hạn chế

Mỗi lời gọi `inc` / `dec` đẩy tối đa 2 vé, nên tổng số vé từng được đẩy không quá `2q`. Mỗi vé bị `heappop` nhiều nhất một lần.

| Thao tác | Chi phí |
|---|---|
| `inc`, `dec` | `O(log q)` (một `heappush` vào heap có tới `O(q)` vé) |
| `getMaxKey`, `getMinKey` | `O(log q)` **khấu hao** (amortized): một lần gọi có thể bỏ nhiều vé (lệnh #9 ở trên bỏ 3 vé), nhưng chia đều cho cả chuỗi lệnh thì mỗi lời gọi tốn trung bình `O(log q)` |
| Bộ nhớ | **`O(q)`** — heap giữ cả vé lỗi thời, lớn theo **lịch sử** lệnh, không theo số key `N` |

Hạn chế:

1. **Không đạt `O(1)`** như đề yêu cầu — vẫn có `log`.
2. **Tốn bộ nhớ theo lịch sử**: một key duy nhất bị `inc` 50 000 lần để lại 50 000 vé trong mỗi heap, dù `N = 1`.
3. Một lời gọi `get` đơn lẻ có thể chậm bất thường (dọn nhiều vé một lúc).

Cách 2 vẫn đáng học: **lazy deletion** là kỹ thuật dùng rất nhiều (Dijkstra, sliding window median, ...). Nhưng để đạt `O(1)` thật sự, ta cần nhìn kỹ hơn vào **cách count thay đổi**.

## 6. Bước đột phá — gom key theo count thành các "bucket"

### 6.1 Quan sát 1: key chỉ nhảy sang count **kề bên**

Heap phải lo cho trường hợp tổng quát "giá trị đổi tùy ý". Nhưng ở bài này count đổi **rất hiền**:

- `inc` chỉ đổi count `c → c + 1`.
- `dec` chỉ đổi count `c → c - 1` (và `1 → 0` thì key biến mất).

Ý tưởng: thay vì lưu "key → count" rồi đi tìm max, hãy **gom các key cùng count** vào một nhóm, gọi là **bucket** (cái xô): bucket count 2 chứa mọi key đang có count 2. Ví dụ sau chuỗi `inc a, inc b, inc b, inc c, inc c, inc c, inc d`:

| count | bucket (set key) |
|---:|---|
| 1 | `{a, d}` |
| 2 | `{b}` |
| 3 | `{c}` |

- Nếu các bucket **xếp theo count tăng dần** thì `getMinKey` = một key trong bucket **đầu**, `getMaxKey` = một key trong bucket **cuối**. Không cần quét!
- `inc b` chỉ chuyển `b` từ bucket 2 sang bucket 3 — bucket **ngay bên phải**. `dec b` chuyển sang bucket **ngay bên trái**.

In [10]:
# Dựng "count -> set key" từ dict đếm, in theo count tăng dần
seq = ["a", "b", "b", "c", "c", "c", "d"]
cnt = {}
for k in seq:
    cnt[k] = cnt.get(k, 0) + 1

groups = {}
for k, c in cnt.items():
    if c not in groups:
        groups[c] = set()
    groups[c].add(k)

for c in sorted(groups):
    print(f"count {c}: {sorted(groups[c])}")
print("min key:", sorted(groups[min(groups)]), "| max key:", sorted(groups[max(groups)]))

count 1: ['a', 'd']
count 2: ['b']
count 3: ['c']
min key: ['a', 'd'] | max key: ['c']


### 6.2 Thử nghiệm: dict `count → set` + hai biến `min_count`, `max_count`

Người mới có thể thử: giữ `groups` (dict `count → set key`) và hai biến số `min_count`, `max_count`. Liệu cập nhật hai biến này có luôn `O(1)`? Xét từng trường hợp khi bucket cũ (count `c`) **trở thành rỗng**:

| Lệnh | Key đi đâu | Cập nhật `max_count` | Cập nhật `min_count` |
|---|---|---|---|
| `inc` key mới | vào count 1 | `max(max_count, 1)` — `O(1)` | `= 1` — `O(1)` |
| `inc`, `c → c+1` | sang count `c+1` | `max(max_count, c+1)` — `O(1)` | nếu bucket `c` rỗng và `c == min_count` → `min_count = c+1` (key vừa tới đó) — `O(1)` |
| `dec`, `c → c-1 ≥ 1` | sang count `c-1` | nếu bucket `c` rỗng và `c == max_count` → `max_count = c-1` — `O(1)` | `min(min_count, c-1)` — `O(1)` |
| `dec`, `1 → 0` | **biến mất** | (chỉ ảnh hưởng nếu mọi key đều count 1) | nếu bucket 1 rỗng → **count nhỏ nhất kế tiếp là bao nhiêu?** ❌ |

Trường hợp cuối là chỗ gãy: key biến mất không "để lại dấu" ở bucket bên cạnh. Count nhỏ nhất kế tiếp có thể ở rất xa. Ví dụ `a` có count 1, `b` có count 1000: sau `dec a`, ta phải dò `2, 3, 4, ...` cho tới `1000`.

In [11]:
groups = {1: {"a"}, 1000: {"b"}}
min_count = 1

# dec("a"): a từ 1 về 0 -> bucket 1 rỗng -> xóa
groups[1].discard("a")
del groups[1]

c, steps = min_count + 1, 1
while c not in groups:          # dò từng count một để tìm bucket không rỗng kế tiếp
    c += 1
    steps += 1
min_count = c
print("min_count mới =", min_count, "| số bước dò =", steps)

min_count mới = 1000 | số bước dò = 999


### 6.3 Kết luận: cần các bucket **xếp theo thứ tự, biết hàng xóm**

Thứ ta thực sự cần là một **dãy các bucket không rỗng, xếp theo count tăng dần**, sao cho:

1. Biết ngay bucket **đầu** (min) và bucket **cuối** (max).
2. Từ một bucket, nhảy ngay sang bucket **kế bên trái / phải** — để biết bucket `c+1` (hoặc `c-1`) đã có sẵn chưa.
3. **Chèn** một bucket mới vào **giữa** dãy (ngay cạnh bucket hiện tại) trong `O(1)`.
4. **Xóa** một bucket rỗng ở **giữa** dãy trong `O(1)` — để không bao giờ phải "dò" như ví dụ trên.

List thường của Python làm tốt 1 và 2, nhưng chèn / xóa ở giữa tốn `O(n)` (phải dời các phần tử phía sau). Cấu trúc làm được **cả 4 điều** trong `O(1)` là **danh sách liên kết đôi** (doubly linked list). Thêm một dict `where: key → bucket đang chứa key` để từ key tìm ra bucket trong `O(1)`, ta có lời giải chính.

## 7. Nền tảng: Danh sách liên kết (linked list)

### 7.1 Danh sách liên kết đơn

Một **node** (nút) là một object nhỏ gồm **dữ liệu** và **con trỏ** `next` trỏ tới node kế tiếp. Các node "nắm tay nhau" thành một chuỗi; node cuối có `next = None`.

```text
 head
  │
  ▼
┌────┬──┐    ┌────┬──┐    ┌────┬──────┐
│ 10 │ ─┼──► │ 20 │ ─┼──► │ 30 │ None │
└────┴──┘    └────┴──┘    └────┴──────┘
  val next
```

Khác list Python: các node **không nằm liền nhau** trong bộ nhớ, ta chỉ đi được bằng cách lần theo `next`. Đổi lại, **chèn một node ngay sau node `p`** chỉ cần đổi 2 con trỏ, không phải dời phần tử nào:

1. `new.next = p.next`  (node mới nắm tay node đứng sau `p`)
2. `p.next = new`       (`p` chuyển sang nắm tay node mới)

Thứ tự quan trọng: nếu làm bước 2 trước, ta mất dấu node đứng sau `p`.

In [12]:
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None


def to_list(head):
    out, cur = [], head
    while cur is not None:          # lần theo next cho tới None
        out.append(cur.val)
        cur = cur.next
    return out


a, b, c = Node(10), Node(20), Node(30)
a.next = b
b.next = c
print(to_list(a))                    # [10, 20, 30]

new = Node(25)                       # chèn 25 ngay sau node b (20): O(1)
new.next = b.next
b.next = new
print(to_list(a))                    # [10, 20, 25, 30]

[10, 20, 30]
[10, 20, 25, 30]


### 7.2 Vì sao cần liên kết **đôi**?

Muốn **xóa** node `x` khỏi danh sách liên kết đơn, ta phải sửa `next` của node **đứng trước** `x`. Nhưng `x` không biết ai đứng trước mình → phải đi từ `head` tìm lại: `O(n)`.

Trong bài toán, ta sẽ có sẵn bucket cần xóa (lấy từ `where[key]`), nên muốn xóa nó ngay mà không phải đi tìm. Giải pháp: mỗi node có **thêm** con trỏ `prev` trỏ về node đứng trước — đó là **danh sách liên kết đôi**:

```text
None ◄── [10] ◄──► [20] ◄──► [30] ──► None
```

Xóa node `x` (đứng giữa `p` và `n`) chỉ cần 2 phép gán:

1. `x.prev.next = x.next`  (node trước bỏ qua `x`, nắm tay node sau)
2. `x.next.prev = x.prev`  (node sau bỏ qua `x`, nắm tay node trước)

Nhưng nếu `x` là node **đầu** (`x.prev is None`) hoặc **cuối** (`x.next is None`) thì hai dòng trên báo lỗi `AttributeError` — phải viết thêm nhiều nhánh `if`. Cell dưới cho thấy chuyện đó.

In [13]:
class DNode:
    def __init__(self, val):
        self.val = val
        self.prev = None
        self.next = None


n10, n20, n30 = DNode(10), DNode(20), DNode(30)
n10.next, n20.prev = n20, n10
n20.next, n30.prev = n30, n20

# Xóa node giữa (20): ổn
n20.prev.next = n20.next
n20.next.prev = n20.prev
cur, vals = n10, []
while cur:
    vals.append(cur.val)
    cur = cur.next
print("sau khi xóa 20:", vals)                       # [10, 30]

# Xóa node đầu (10) bằng đúng 2 dòng đó -> lỗi vì n10.prev là None
try:
    n10.prev.next = n10.next
except AttributeError as e:
    print("xóa node đầu không có sentinel -> AttributeError:", e)

sau khi xóa 20: [10, 30]
xóa node đầu không có sentinel -> AttributeError: 'NoneType' object has no attribute 'next' and no __dict__ for setting new attributes


### 7.3 Sentinel `head` / `tail`: hai node "giả" canh hai đầu

Mẹo kinh điển: tạo sẵn **hai node giả** (sentinel — lính canh) `head` và `tail` không chứa dữ liệu thật, đứng ở hai đầu danh sách **mãi mãi**. Danh sách rỗng trông như thế này:

```text
HEAD ◄──► TAIL          (head.next is tail, tail.prev is head)
```

Mọi node **thật** luôn nằm **giữa** `head` và `tail`, nên node thật nào cũng có `prev` và `next` khác `None`. Nhờ vậy:

- **Chèn sau node `p`** (kể cả `p = head`): 4 phép gán, không cần `if`.
  1. `new.prev = p`
  2. `new.next = p.next`
  3. `p.next.prev = new`
  4. `p.next = new`  ← làm **sau cùng**, vì bước 2, 3 còn cần `p.next` cũ.
- **Xóa node `x`** (kể cả node thật đầu tiên / cuối cùng): 2 phép gán như mục 7.2, không cần `if`.
- **Node thật đầu tiên** = `head.next`; **node thật cuối cùng** = `tail.prev`; **rỗng** ⇔ `head.next is tail`.

Cell dưới cài một lớp `DLL` (doubly linked list) nhỏ có sentinel và kiểm tra **cả hai chiều** (đi xuôi theo `next`, đi ngược theo `prev`) để chắc mọi con trỏ đều được nối đúng.

In [14]:
class DLL:
    def __init__(self):
        self.head = DNode("HEAD")          # sentinel đầu
        self.tail = DNode("TAIL")          # sentinel cuối
        self.head.next = self.tail
        self.tail.prev = self.head

    def insert_after(self, p, val):
        new = DNode(val)
        new.prev = p
        new.next = p.next
        p.next.prev = new
        p.next = new                       # làm sau cùng
        return new

    def remove(self, x):
        x.prev.next = x.next
        x.next.prev = x.prev

    def forward(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.val)
            cur = cur.next
        return out

    def backward(self):
        out, cur = [], self.tail.prev
        while cur is not self.head:
            out.append(cur.val)
            cur = cur.prev
        return out


d = DLL()
print("rỗng:", d.forward(), d.head.next is d.tail)
x10 = d.insert_after(d.head, 10)           # chèn ngay sau head = thêm vào đầu
x30 = d.insert_after(x10, 30)
x20 = d.insert_after(x10, 20)              # chèn vào GIỮA 10 và 30
print("xuôi:", d.forward(), "| ngược:", d.backward())

d.remove(x10)                              # xóa node thật đầu tiên: không cần if
d.remove(x30)                              # xóa node thật cuối cùng: không cần if
print("xuôi:", d.forward(), "| ngược:", d.backward())
d.remove(x20)
print("rỗng lại:", d.forward(), d.head.next is d.tail, d.tail.prev is d.head)

rỗng: [] True
xuôi: [10, 20, 30] | ngược: [30, 20, 10]
xuôi: [20] | ngược: [20]
rỗng lại: [] True True


## 8. Cách 3 — Danh sách liên kết đôi các bucket + dict (cách chính, `O(1)`)

### 8.1 Thiết kế

Mỗi **bucket** là một node của danh sách liên kết đôi, gồm:

- `count`: số đếm chung của các key trong bucket;
- `keys`: `set` các key đang có đúng count này;
- `prev`, `next`: con trỏ sang bucket kề bên.

Object `AllOne` giữ:

- `head`, `tail`: hai sentinel (count giả `0`, `keys` rỗng, không bao giờ bị xóa);
- `where`: dict `key → bucket đang chứa key`.

Ví dụ trạng thái `cnt = {a: 1, d: 1, b: 2, c: 5}`:

```text
┌────────┐    ┌────────────┐    ┌────────────┐    ┌────────────┐    ┌────────┐
│  HEAD  │ ──►│ count = 1  │ ──►│ count = 2  │ ──►│ count = 5  │ ──►│  TAIL  │
│ (giả)  │◄── │ keys={a,d} │◄── │ keys={b}   │◄── │ keys={c}   │◄── │ (giả)  │
└────────┘    └────────────┘    └────────────┘    └────────────┘    └────────┘
where = {a: bucket1, d: bucket1, b: bucket2, c: bucket5}
```

Viết gọn một dòng (cách notebook sẽ vẽ): `HEAD <-> [1:{a,d}] <-> [2:{b}] <-> [5:{c}] <-> TAIL`.

**Bốn invariant** (bất biến) luôn đúng sau mỗi lệnh:

1. Đi từ `head` sang `tail`, count các bucket **tăng ngặt** (không có hai bucket cùng count).
2. Không có bucket thật nào **rỗng** (bucket rỗng bị xóa ngay).
3. Mỗi key đang tồn tại nằm trong **đúng một** bucket, và bucket đó có `count` = count thật của key.
4. `where[key]` trỏ đúng tới bucket chứa `key`; key đã về 0 thì không còn trong `where`.

Từ 1 và 2: `getMinKey` = một key bất kỳ của `head.next`, `getMaxKey` = một key bất kỳ của `tail.prev`; rỗng ⇔ `head.next is tail`.

### 8.2 `inc(key)` — 3 bước

Gọi `b` là bucket hiện tại của key và `c` là count hiện tại. Nếu key **mới**, coi như `b = head` và `c = 0` (sentinel `head` đóng vai "bucket count 0" — mẹo làm code gọn).

**Bước 1 — tìm hoặc tạo bucket `c + 1` ngay bên phải `b`.** Vì các bucket tăng ngặt và key chỉ lên đúng 1 nấc, bucket `c + 1` nếu có thì **phải** là `b.next`:

- Nếu `b.next` là bucket thật có `count == c + 1` → dùng luôn.
- Ngược lại (`b.next` là `tail`, hoặc `b.next.count > c + 1`) → **tạo** bucket mới `c + 1` và chèn ngay sau `b`. Thứ tự vẫn tăng ngặt vì `c < c + 1 < b.next.count`.

**Bước 2 — chuyển key:** thêm key vào bucket `c + 1`, cập nhật `where[key]`.

**Bước 3 — dọn bucket cũ:** nếu `b` không phải `head`, bỏ key khỏi `b.keys`; nếu `b` trở thành rỗng → **xóa** `b` khỏi danh sách.

Chú ý thứ tự: phải **chèn bucket mới (bước 1) trước, rồi mới xóa `b` (bước 3)** — vì ta chèn "ngay sau `b`", cần `b` còn nằm trong danh sách. (Mục 12 có demo lỗi khi làm ngược.)

```text
Ví dụ: HEAD <-> [1:{a}] <-> [3:{b}] <-> TAIL, gọi inc("a")
  b = [1:{a}], c = 1, b.next = [3]  -> count 3 ≠ 2 -> TẠO [2] giữa [1] và [3]
  HEAD <-> [1:{a}] <-> [2:{}] <-> [3:{b}] <-> TAIL
  thêm a vào [2], bỏ a khỏi [1] -> [1] rỗng -> XÓA
  HEAD <-> [2:{a}] <-> [3:{b}] <-> TAIL
```

### 8.3 `dec(key)` — đối xứng với `inc`

Gọi `b = where[key]`, `c = b.count`.

**Bước 1 — chuyển key sang count `c - 1`:**

- Nếu `c == 1`: key về 0 → **xóa** khỏi `where`, không đưa vào bucket nào (tuyệt đối không tạo "bucket count 0").
- Nếu `c > 1`: bucket `c - 1` nếu có thì **phải** là `b.prev`. Nếu `b.prev` là bucket thật có `count == c - 1` → dùng luôn; ngược lại (`b.prev` là `head` hoặc `b.prev.count < c - 1`) → **tạo** bucket `c - 1` chèn ngay **trước** `b`, tức là `insert_after(b.prev, c - 1)`. Thêm key vào đó, cập nhật `where[key]`.

**Bước 2 — dọn bucket cũ:** bỏ key khỏi `b.keys`; nếu `b` rỗng → xóa `b`.

Đề bảo đảm `dec` chỉ được gọi với key đang tồn tại, nên `where[key]` luôn có.

```text
Ví dụ: HEAD <-> [1:{c}] <-> [3:{a,b}] <-> TAIL, gọi dec("b")
  b = [3:{a,b}], c = 3, b.prev = [1] -> count 1 ≠ 2 -> TẠO [2] giữa [1] và [3]
  thêm b vào [2], bỏ b khỏi [3] -> [3] còn {a} -> giữ lại
  HEAD <-> [1:{c}] <-> [2:{b}] <-> [3:{a}] <-> TAIL
```

### 8.4 `getMaxKey` / `getMinKey`

- Rỗng (`head.next is tail`) → `""`.
- Ngược lại: `next(iter(self.tail.prev.keys))` cho max, `next(iter(self.head.next.keys))` cho min.

In [15]:
class Bucket:
    def __init__(self, count):
        self.count = count
        self.keys = set()
        self.prev = None
        self.next = None


class AllOneBuckets:
    # Cách 3: danh sách liên kết đôi các bucket (count tăng dần) + dict key -> bucket
    def __init__(self):
        self.head = Bucket(0)               # sentinel đầu, đóng vai "count 0"
        self.tail = Bucket(0)               # sentinel cuối
        self.head.next = self.tail
        self.tail.prev = self.head
        self.where = {}                     # key -> bucket đang chứa key

    # ---- 2 thao tác con trỏ cơ bản ----
    def _insert_after(self, node, count):
        new = Bucket(count)
        new.prev = node
        new.next = node.next
        node.next.prev = new
        node.next = new
        return new

    def _remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    # ---- 4 method của đề ----
    def inc(self, key: str) -> None:
        b = self.where.get(key, self.head)  # key mới -> coi như đang ở head (count 0)
        c = b.count
        nxt = b.next
        if nxt is self.tail or nxt.count != c + 1:
            nxt = self._insert_after(b, c + 1)      # bước 1: tạo bucket c+1 ngay sau b
        nxt.keys.add(key)                           # bước 2: chuyển key
        self.where[key] = nxt
        if b is not self.head:                      # bước 3: dọn bucket cũ
            b.keys.remove(key)
            if not b.keys:
                self._remove(b)

    def dec(self, key: str) -> None:
        b = self.where[key]
        c = b.count
        if c == 1:
            del self.where[key]                     # về 0 -> biến mất
        else:
            prv = b.prev
            if prv is self.head or prv.count != c - 1:
                prv = self._insert_after(b.prev, c - 1)   # tạo bucket c-1 ngay trước b
            prv.keys.add(key)
            self.where[key] = prv
        b.keys.remove(key)
        if not b.keys:
            self._remove(b)

    def getMaxKey(self) -> str:
        if self.tail.prev is self.head:
            return ""
        return next(iter(self.tail.prev.keys))

    def getMinKey(self) -> str:
        if self.head.next is self.tail:
            return ""
        return next(iter(self.head.next.keys))

    # ---- tiện ích để học: liệt kê và vẽ các bucket ----
    def buckets(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append((cur.count, sorted(cur.keys)))
            cur = cur.next
        return out

    def draw(self):
        parts = ["HEAD"] + [f"[{c}:{{{','.join(ks)}}}]" for c, ks in self.buckets()] + ["TAIL"]
        return " <-> ".join(parts)


obj = AllOneBuckets()
print(obj.draw())
for k in ["a", "d", "b", "b", "c", "c", "c", "c", "c"]:
    obj.inc(k)
print(obj.draw())                           # giống hình ở mục 8.1
print("max:", obj.getMaxKey(), "| min là a hoặc d:", obj.getMinKey() in {"a", "d"})

HEAD <-> TAIL
HEAD <-> [1:{a,d}] <-> [2:{b}] <-> [5:{c}] <-> TAIL
max: c | min là a hoặc d: True


In [16]:
# Example 1 của đề, vẽ danh sách bucket sau mỗi lệnh
obj = AllOneBuckets()
for op, a in zip(EX1_OPS[1:], EX1_ARGS[1:]):
    res = getattr(obj, op)(*a)
    print(f"{op}({', '.join(map(repr, a))})".ljust(18), "->", repr(res).ljust(8), "|", obj.draw())

got = run_commands(AllOneBuckets, EX1_OPS, EX1_ARGS)
print(show(got))
assert got == EX1_OUT

inc('hello')       -> None     | HEAD <-> [1:{hello}] <-> TAIL
inc('hello')       -> None     | HEAD <-> [2:{hello}] <-> TAIL
getMaxKey()        -> 'hello'  | HEAD <-> [2:{hello}] <-> TAIL
getMinKey()        -> 'hello'  | HEAD <-> [2:{hello}] <-> TAIL
inc('leet')        -> None     | HEAD <-> [1:{leet}] <-> [2:{hello}] <-> TAIL
getMaxKey()        -> 'hello'  | HEAD <-> [1:{leet}] <-> [2:{hello}] <-> TAIL
getMinKey()        -> 'leet'   | HEAD <-> [1:{leet}] <-> [2:{hello}] <-> TAIL
[null, null, null, "hello", "hello", null, "hello", "leet"]


### 8.5 Trace kỹ: khi nào tạo bucket, khi nào xóa bucket

Chuỗi lệnh được chọn để đi qua **mọi nhánh**: tạo bucket ở cuối, ở đầu, ở **giữa**; gộp vào bucket có sẵn; xóa bucket rỗng ở đầu / giữa / cuối; key về 0; hòa max/min; về rỗng.

| # | Lệnh | Count của key | Tạo / xóa bucket | Danh sách sau lệnh |
|---:|---|---|---|---|
| 1 | `inc a` | `a: 0→1` | tạo `1` (sau `HEAD`) | `[1:{a}]` |
| 2 | `inc b` | `b: 0→1` | gộp vào `1` có sẵn | `[1:{a,b}]` |
| 3 | `inc b` | `b: 1→2` | tạo `2` sau `[1]` (vì `next` là `TAIL`) | `[1:{a}] [2:{b}]` |
| 4 | `inc b` | `b: 2→3` | tạo `3`; `[2]` rỗng → xóa | `[1:{a}] [3:{b}]` |
| 5 | `inc a` | `a: 1→2` | `next` là `[3]` ≠ 2 → tạo `2` **ở giữa**; `[1]` rỗng → xóa | `[2:{a}] [3:{b}]` |
| 6 | `inc c` | `c: 0→1` | `head.next` là `[2]` ≠ 1 → tạo `1` **ở đầu** | `[1:{c}] [2:{a}] [3:{b}]` |
| 7 | `getMaxKey` | | | trả `"b"` (count 3) |
| 8 | `getMinKey` | | | trả `"c"` (count 1) |
| 9 | `dec b` | `b: 3→2` | gộp vào `[2]` (là `prev`); `[3]` rỗng → xóa **ở cuối** | `[1:{c}] [2:{a,b}]` |
| 10 | `getMaxKey` | | | hòa: `"a"` **hoặc** `"b"` (count 2) |
| 11 | `dec c` | `c: 1→0` | `c` biến mất; `[1]` rỗng → xóa **ở đầu** | `[2:{a,b}]` |
| 12 | `dec a` | `a: 2→1` | `prev` là `HEAD` → tạo `1` trước `[2]` | `[1:{a}] [2:{b}]` |
| 13 | `getMinKey` | | | trả `"a"` |
| 14 | `inc a` | `a: 1→2` | gộp vào `[2]`; `[1]` rỗng → xóa | `[2:{a,b}]` |
| 15 | `dec b` | `b: 2→1` | tạo `1` trước `[2]` | `[1:{b}] [2:{a}]` |
| 16 | `dec b` | `b: 1→0` | `b` biến mất; `[1]` rỗng → xóa | `[2:{a}]` |
| 17 | `dec a` | `a: 2→1` | tạo `1`; `[2]` rỗng → xóa | `[1:{a}]` |
| 18 | `dec a` | `a: 1→0` | `[1]` rỗng → xóa | rỗng |
| 19 | `getMaxKey` | | | `""` |
| 20 | `getMinKey` | | | `""` |

Hàm `trace_buckets` dưới đây tự suy ra cột "Tạo / xóa bucket" bằng cách so danh sách count **trước** và **sau** mỗi lệnh. Với lệnh hỏi có hòa, key in ra có thể khác nhau giữa các lần chạy (thứ tự duyệt `set`) — điều quan trọng là **count** của nó đúng.

In [17]:
def trace_buckets(obj, commands, start=1):
    # commands: list (op, key) với key = None cho getMaxKey/getMinKey
    for i, (op, key) in enumerate(commands, start=start):
        if op in ("inc", "dec"):
            old = obj.where[key].count if key in obj.where else 0
            before = [c for c, _ in obj.buckets()]
            getattr(obj, op)(key)
            after = [c for c, _ in obj.buckets()]
            new = obj.where[key].count if key in obj.where else 0
            notes = [f"tạo {c}" for c in after if c not in before]
            notes += [f"xóa {c}" for c in before if c not in after]
            print(f"#{i:<2} {op}({key!r}): {key} {old}->{new} | {', '.join(notes) or 'không tạo/xóa'}")
            print("     " + obj.draw())
        else:
            res = getattr(obj, op)()
            c = obj.where[res].count if res else None
            print(f"#{i:<2} {op}() -> {res!r}" + (f" (count {c})" if res else ""))


PART1 = [("inc", "a"), ("inc", "b"), ("inc", "b"), ("inc", "b"), ("inc", "a"),
         ("inc", "c"), ("getMaxKey", None), ("getMinKey", None)]
PART2 = [("dec", "b"), ("getMaxKey", None), ("dec", "c"), ("dec", "a"), ("getMinKey", None),
         ("inc", "a"), ("dec", "b"), ("dec", "b"), ("dec", "a"), ("dec", "a"),
         ("getMaxKey", None), ("getMinKey", None)]

tr = AllOneBuckets()
trace_buckets(tr, PART1)

#1  inc('a'): a 0->1 | tạo 1
     HEAD <-> [1:{a}] <-> TAIL
#2  inc('b'): b 0->1 | không tạo/xóa
     HEAD <-> [1:{a,b}] <-> TAIL
#3  inc('b'): b 1->2 | tạo 2
     HEAD <-> [1:{a}] <-> [2:{b}] <-> TAIL
#4  inc('b'): b 2->3 | tạo 3, xóa 2
     HEAD <-> [1:{a}] <-> [3:{b}] <-> TAIL
#5  inc('a'): a 1->2 | tạo 2, xóa 1
     HEAD <-> [2:{a}] <-> [3:{b}] <-> TAIL
#6  inc('c'): c 0->1 | tạo 1
     HEAD <-> [1:{c}] <-> [2:{a}] <-> [3:{b}] <-> TAIL
#7  getMaxKey() -> 'b' (count 3)
#8  getMinKey() -> 'c' (count 1)


In [18]:
trace_buckets(tr, PART2, start=len(PART1) + 1)   # tiếp tục trên cùng object tr

#9  dec('b'): b 3->2 | xóa 3
     HEAD <-> [1:{c}] <-> [2:{a,b}] <-> TAIL
#10 getMaxKey() -> 'a' (count 2)
#11 dec('c'): c 1->0 | xóa 1
     HEAD <-> [2:{a,b}] <-> TAIL
#12 dec('a'): a 2->1 | tạo 1
     HEAD <-> [1:{a}] <-> [2:{b}] <-> TAIL
#13 getMinKey() -> 'a' (count 1)
#14 inc('a'): a 1->2 | xóa 1
     HEAD <-> [2:{a,b}] <-> TAIL
#15 dec('b'): b 2->1 | tạo 1
     HEAD <-> [1:{b}] <-> [2:{a}] <-> TAIL
#16 dec('b'): b 1->0 | xóa 1
     HEAD <-> [2:{a}] <-> TAIL
#17 dec('a'): a 2->1 | tạo 1, xóa 2
     HEAD <-> [1:{a}] <-> TAIL
#18 dec('a'): a 1->0 | xóa 1
     HEAD <-> TAIL
#19 getMaxKey() -> ''
#20 getMinKey() -> ''


### 8.6 Đọc lại trace — những điểm đáng chú ý

- **#3 và #4**: `inc b` hai lần liên tiếp. Lần #3 bucket `[1]` vẫn còn `a` nên **giữ lại**; lần #4 bucket `[2]` chỉ có `b` nên sau khi `b` rời đi nó rỗng và bị **xóa** — danh sách nhảy từ `[1] [2]` sang `[1] [3]`. Đó chính là lý do count trong danh sách có thể "thưa" (1, 3) nhưng luôn **tăng ngặt**.
- **#5**: ca khó nhất của `inc`: `b.next` tồn tại nhưng count là 3, không phải 2 → phải **chèn vào giữa**. Nếu code lười "dùng luôn `b.next`" thì `a` sẽ bị đưa nhầm lên count 3 (xem lỗi 4 ở mục 12).
- **#6**: key mới, `b = head`. `head.next` là `[2]` chứ không phải `[1]` → tạo `[1]` ngay sau `head`. Nhờ coi `head` là "bucket count 0", key mới dùng **chung** code với key cũ.
- **#9**: `dec` gộp vào bucket bên trái có sẵn, bucket cũ ở **cuối** bị xóa → `tail.prev` tự động thành bucket max mới. Không cần "dò tìm max mới" như mục 6.2!
- **#11**: ca từng làm gãy thử nghiệm ở 6.2: key ở count **min** về 0. Bucket `[1]` bị xóa → `head.next` tự động là `[2]`, min mới có ngay trong `O(1)`. Danh sách liên kết đã "nhớ sẵn hàng xóm" cho ta.
- **#18 → #19, #20**: xóa bucket cuối cùng → `head.next is tail` → trả `""`.

### 8.7 Vì sao cách 3 luôn đúng?

Ta kiểm tra **4 invariant** ở mục 8.1 được giữ sau mỗi `inc` / `dec` (lúc đầu danh sách rỗng nên đúng hiển nhiên):

- **Tăng ngặt (1)**: `inc` chỉ tạo bucket `c + 1` ngay sau `b` (count `c`) khi `b.next.count ≠ c + 1`. Vì trước đó `b.next.count > c` (tăng ngặt), nên lúc này `b.next.count > c + 1` (hoặc `b.next` là `tail`) → chèn `c + 1` vào giữa vẫn tăng ngặt. `dec` đối xứng: chỉ tạo `c - 1` ngay trước `b` khi `b.prev.count < c - 1` (hoặc `b.prev` là `head`). Xóa một bucket không phá thứ tự.
- **Không rỗng (2)**: bucket mới tạo ra **luôn** được thêm ngay một key; bucket duy nhất có thể mất key là `b`, và ta xóa nó nếu rỗng.
- **Đúng count (3) và `where` đúng (4)**: chỉ đúng một key đổi count mỗi lệnh; key đó được chuyển từ `b` sang đúng bucket count mới (hoặc bị xóa khỏi `where` khi về 0), mọi key khác giữ nguyên chỗ.

Từ đó: bucket đầu chứa đúng các key có count **nhỏ nhất**, bucket cuối chứa các key có count **lớn nhất** → `getMinKey` / `getMaxKey` đúng; rỗng ⇔ không còn key nào → trả `""`.

Cell dưới kiểm chứng invariant **sau từng lệnh** trên hàng nghìn lệnh ngẫu nhiên, so count với một dict đếm độc lập và duyệt danh sách **cả xuôi lẫn ngược** (bắt lỗi nối thiếu `prev`).

In [19]:
import random


def check_invariants(obj, true_cnt):
    fwd, cur = [], obj.head.next
    while cur is not obj.tail:
        fwd.append(cur)
        cur = cur.next
    bwd, cur = [], obj.tail.prev
    while cur is not obj.head:
        bwd.append(cur)
        cur = cur.prev
    assert fwd == bwd[::-1], "con trỏ prev/next không khớp"
    counts = [b.count for b in fwd]
    assert all(x < y for x, y in zip(counts, counts[1:])), "count không tăng ngặt"
    assert all(b.keys for b in fwd), "có bucket rỗng"
    seen = {}
    for b in fwd:
        for k in b.keys:
            assert k not in seen, "key nằm ở 2 bucket"
            seen[k] = b.count
            assert obj.where[k] is b, "where sai"
    assert seen == true_cnt and set(obj.where) == set(true_cnt), "count lệch"


random.seed(432)
total = 0
for trial in range(200):
    obj, true_cnt = AllOneBuckets(), {}
    for _ in range(random.randint(1, 60)):
        if true_cnt and random.random() < 0.45:
            k = random.choice(sorted(true_cnt))
            obj.dec(k)
            true_cnt[k] -= 1
            if true_cnt[k] == 0:
                del true_cnt[k]
        else:
            k = random.choice("abcde")
            obj.inc(k)
            true_cnt[k] = true_cnt.get(k, 0) + 1
        check_invariants(obj, true_cnt)
        total += 1
print("invariant đúng sau", total, "lệnh ngẫu nhiên")

invariant đúng sau 6338 lệnh ngẫu nhiên


### 8.8 Độ phức tạp cách 3

Mỗi lời gọi chỉ làm một **số cố định** thao tác:

| Thao tác | Gồm những gì | Chi phí |
|---|---|---|
| `inc` | 1 lần tra `where`, tối đa 1 lần chèn bucket (4 phép gán con trỏ), 1 `add` + 1 `remove` trên set, tối đa 1 lần xóa bucket (2 phép gán) | `O(1)` trung bình |
| `dec` | tương tự | `O(1)` trung bình |
| `getMaxKey`, `getMinKey` | 1 phép so sánh sentinel + `next(iter(set))` | `O(1)` |
| Bộ nhớ | mỗi key nằm 1 chỗ trong `where` và 1 chỗ trong một bucket; số bucket ≤ `N` (mỗi bucket có ≥ 1 key) | `O(N)` |

- Chữ "trung bình" đến từ `dict` / `set` (hash table). Key dài tối đa 10 ký tự nên tính hash coi như `O(1)`.
- Các thao tác con trỏ `prev` / `next` là `O(1)` **trong mọi trường hợp**.
- Ghi chú cho người tò mò: trong CPython, `next(iter(s))` duyệt bảng hash từ đầu tới ô có dữ liệu đầu tiên. Nếu một set từng rất lớn rồi bị xóa gần hết, bước này có thể phải lướt qua nhiều ô trống. Trên thực tế rất nhanh và được LeetCode chấp nhận; nếu muốn chắc chắn tuyệt đối, có thể thay `set` bằng một danh sách liên kết các key.

So với cách 2: bộ nhớ chỉ phụ thuộc **số key hiện có** (`N`), không phụ thuộc lịch sử lệnh (`q`).

## 9. Code nộp LeetCode

Bài thiết kế không dùng `class Solution`: LeetCode tạo trực tiếp object `AllOne()` rồi gọi các method. Ta nộp đúng tên class `AllOne` và đúng chữ ký các method. Được phép khai báo thêm class phụ (`BucketNode`) phía trên. Thuộc tính `__slots__` là tùy chọn — nó báo trước cho Python các thuộc tính của node, giúp node nhẹ và nhanh hơn một chút.

Bản nộp giống `AllOneBuckets` nhưng bỏ phần tiện ích vẽ hình.

In [20]:
class BucketNode:
    __slots__ = ("count", "keys", "prev", "next")

    def __init__(self, count):
        self.count = count
        self.keys = set()
        self.prev = None
        self.next = None


class AllOne:

    def __init__(self):
        self.head = BucketNode(0)          # sentinel đầu (count 0)
        self.tail = BucketNode(0)          # sentinel cuối
        self.head.next = self.tail
        self.tail.prev = self.head
        self.where = {}                    # key -> bucket chứa key

    def _insert_after(self, node, count):
        new = BucketNode(count)
        new.prev, new.next = node, node.next
        node.next.prev = new
        node.next = new
        return new

    def _remove(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def inc(self, key: str) -> None:
        b = self.where.get(key, self.head)
        nxt = b.next
        if nxt is self.tail or nxt.count != b.count + 1:
            nxt = self._insert_after(b, b.count + 1)
        nxt.keys.add(key)
        self.where[key] = nxt
        if b is not self.head:
            b.keys.remove(key)
            if not b.keys:
                self._remove(b)

    def dec(self, key: str) -> None:
        b = self.where[key]
        if b.count == 1:
            del self.where[key]
        else:
            prv = b.prev
            if prv is self.head or prv.count != b.count - 1:
                prv = self._insert_after(b.prev, b.count - 1)
            prv.keys.add(key)
            self.where[key] = prv
        b.keys.remove(key)
        if not b.keys:
            self._remove(b)

    def getMaxKey(self) -> str:
        return "" if self.tail.prev is self.head else next(iter(self.tail.prev.keys))

    def getMinKey(self) -> str:
        return "" if self.head.next is self.tail else next(iter(self.head.next.keys))


# Your AllOne object will be instantiated and called as such:
# obj = AllOne()
# obj.inc(key)
# obj.dec(key)
# param_3 = obj.getMaxKey()
# param_4 = obj.getMinKey()

### 9.1 Bộ test cho `AllOne`

Vì đề cho phép trả "một trong các key" khi hòa, ta viết hàm `valid_outputs(ops, args, got)`: tự mô phỏng một dict đếm độc lập theo đúng chuỗi lệnh, rồi với mỗi `getMaxKey` / `getMinKey` kiểm tra:

- dict rỗng → output phải là `""`;
- ngược lại → output phải là một key đang tồn tại và **count của nó** bằng count lớn nhất / nhỏ nhất.

Các ca test: Example 1 (so **nguyên văn**, vì không có hòa), hỏi khi rỗng ngay từ đầu, `dec` về 0 rồi rỗng, nhiều key hòa, key quay lại sau khi đã bị xóa, chuỗi "lên cao rồi xuống" tạo bucket giữa.

In [21]:
def valid_outputs(ops, args, got):
    cnt = {}
    for op, a, out in zip(ops[1:], args[1:], got[1:]):
        if op == "inc":
            cnt[a[0]] = cnt.get(a[0], 0) + 1
            assert out is None
        elif op == "dec":
            cnt[a[0]] -= 1
            if cnt[a[0]] == 0:
                del cnt[a[0]]
            assert out is None
        else:
            if not cnt:
                assert out == "", (op, out)
            else:
                best = max(cnt.values()) if op == "getMaxKey" else min(cnt.values())
                assert out in cnt and cnt[out] == best, (op, out, cnt)
    return True


def L(*tokens):
    # Viết test gọn: "+a" -> inc a, "-a" -> dec a, "max"/"min" -> getMaxKey/getMinKey
    ops, args = ["AllOne"], [[]]
    for t in tokens:
        if t == "max":
            ops.append("getMaxKey"); args.append([])
        elif t == "min":
            ops.append("getMinKey"); args.append([])
        else:
            ops.append("inc" if t[0] == "+" else "dec"); args.append([t[1:]])
    return ops, args


# 1) Example 1 nguyên văn
got = run_commands(AllOne, EX1_OPS, EX1_ARGS)
assert got == EX1_OUT
print("Example 1:", show(got))

tests = {
    "rỗng từ đầu":           L("max", "min"),
    "dec về 0 rồi rỗng":     L("+a", "+a", "-a", "max", "-a", "max", "min"),
    "nhiều key hòa":         L("+a", "+b", "+c", "max", "min", "+b", "+c", "max", "min"),
    "key quay lại sau xóa":  L("+a", "-a", "min", "+a", "+a", "max", "min"),
    "lên cao rồi xuống":     L("+a", "+a", "+a", "+b", "-a", "-a", "max", "min", "-b", "max", "min"),
    "test LeetCode quen":    L("+a", "+b", "+b", "+b", "+b", "-b", "-b", "max", "min"),
}
for name, (ops, args) in tests.items():
    got = run_commands(AllOne, ops, args)
    assert valid_outputs(ops, args, got)
    print(f"{name:<22} PASS  {show(got)}")

Example 1: [null, null, null, "hello", "hello", null, "hello", "leet"]
rỗng từ đầu            PASS  [null, "", ""]
dec về 0 rồi rỗng      PASS  [null, null, null, null, "a", null, "", ""]
nhiều key hòa          PASS  [null, null, null, null, "a", "a", null, null, "c", "a"]
key quay lại sau xóa   PASS  [null, null, null, "", null, null, "a", "a"]
lên cao rồi xuống      PASS  [null, null, null, null, null, null, null, "a", "a", null, "a", "a"]
test LeetCode quen     PASS  [null, null, null, null, null, null, null, null, "b", "a"]


## 10. Test đối chiếu chéo trên chuỗi lệnh ngẫu nhiên

Ta chạy **song song** (lockstep — cùng nhịp) cách 1 `AllOneScan` (baseline, chắc chắn đúng) với các cách còn lại trên cùng một chuỗi lệnh ngẫu nhiên. Ở mỗi lệnh `getMaxKey` / `getMinKey`:

- nếu cách 1 trả `""` thì cách kia cũng phải trả `""`;
- ngược lại, **không** so chuỗi (vì có thể hòa) mà so **count**: `scan.cnt[kết quả cách kia] == scan.cnt[kết quả cách 1]`.

`random.seed(...)` cố định để mỗi lần chạy ra cùng dữ liệu, có lỗi thì tái hiện được. `random_commands` chỉ sinh `dec` cho key **đang tồn tại** (đúng constraint). Có 4 kịch bản: ít key (rất nhiều hòa, hay về rỗng), 5 key, 50 key ngẫu nhiên dài 1–10 chữ, và kịch bản "dec nhiều" để key liên tục về 0. Cuối cùng là một test cỡ lớn đúng `5·10^4` lời gọi.

In [22]:
import string


def random_key():
    return "".join(random.choice(string.ascii_lowercase) for _ in range(random.randint(1, 10)))


def random_commands(num_calls, pool, p_inc=0.45, p_dec=0.25):
    cmds, cnt = [], {}
    for _ in range(num_calls):
        r = random.random()
        if r < p_dec and cnt:                          # dec chỉ trên key đang có
            k = random.choice(sorted(cnt))
            cmds.append(("dec", k))
            cnt[k] -= 1
            if cnt[k] == 0:
                del cnt[k]
        elif r < p_dec + p_inc or not cnt and r < p_dec:
            k = random.choice(pool)
            cmds.append(("inc", k))
            cnt[k] = cnt.get(k, 0) + 1
        else:
            cmds.append((random.choice(["getMaxKey", "getMinKey"]), None))
    return cmds


def cross_check(cmds, classes):
    ref, objs = AllOneScan(), [cls() for cls in classes]
    for op, key in cmds:
        if key is not None:
            getattr(ref, op)(key)
            for o in objs:
                getattr(o, op)(key)
        else:
            r1 = getattr(ref, op)()
            for cls, o in zip(classes, objs):
                r = getattr(o, op)()
                if r1 == "":
                    assert r == "", (cls.__name__, op, r)
                else:
                    assert ref.cnt.get(r) == ref.cnt[r1], (cls.__name__, op, r, r1)


CLASSES = [AllOneLazyHeap, AllOneBuckets, AllOne]
random.seed(4320)
scenarios = [
    ("2 key 'a','b'", lambda: ["a", "b"], 0.45, 0.25),
    ("5 key", lambda: list("abcde"), 0.45, 0.25),
    ("50 key ngẫu nhiên", lambda: [random_key() for _ in range(50)], 0.45, 0.25),
    ("dec nhiều", lambda: list("abcdef"), 0.40, 0.40),
]
for title, make_pool, p_inc, p_dec in scenarios:
    for _ in range(300):
        cmds = random_commands(random.randint(1, 80), make_pool(), p_inc, p_dec)
        cross_check(cmds, CLASSES)
    print(f"{title:<20}: {len(CLASSES)} cách x 300 chuỗi lệnh -> khớp cách 1 (so theo count)")

# Test cỡ lớn: 5·10^4 lời gọi, 300 key
pool = [random_key() for _ in range(300)]
big = random_commands(5 * 10**4, pool)
cross_check(big, CLASSES)
print(f"cỡ lớn: {len(big)} lời gọi, {sum(k is None for _, k in big)} lần hỏi -> khớp")

ops_big = ["AllOne"] + [op for op, _ in big]
args_big = [[]] + [[k] if k is not None else [] for _, k in big]
assert valid_outputs(ops_big, args_big, run_commands(AllOne, ops_big, args_big))
print("AllOne (nộp bài) qua valid_outputs trên test cỡ lớn")

2 key 'a','b'       : 3 cách x 300 chuỗi lệnh -> khớp cách 1 (so theo count)


5 key               : 3 cách x 300 chuỗi lệnh -> khớp cách 1 (so theo count)


50 key ngẫu nhiên   : 3 cách x 300 chuỗi lệnh -> khớp cách 1 (so theo count)


dec nhiều           : 3 cách x 300 chuỗi lệnh -> khớp cách 1 (so theo count)


cỡ lớn: 50000 lời gọi, 14781 lần hỏi -> khớp
AllOne (nộp bài) qua valid_outputs trên test cỡ lớn


### 10.1 Đo thời gian: kịch bản xấu nhất cho cách 1

Kịch bản: `inc` 3000 key khác nhau, rồi hỏi xen kẽ `getMaxKey` / `getMinKey` 6000 lần. Cách 1 phải quét 3000 key **mỗi** lần hỏi (≈ `1.8·10^7` bước); cách 3 chỉ nhìn `head.next` / `tail.prev`. Số đo phụ thuộc máy, chỉ cần để ý **tỉ lệ** chênh lệch.

Phần sau của cell cho thấy hạn chế bộ nhớ của cách 2: một key duy nhất được `inc` `5·10^4` lần.

In [23]:
import time


def bench(cls, cmds):
    obj = cls()
    t0 = time.perf_counter()
    for op, key in cmds:
        if key is None:
            getattr(obj, op)()
        else:
            getattr(obj, op)(key)
    return time.perf_counter() - t0


random.seed(1)
keys = [f"k{i}" for i in range(3000)]
worst = [("inc", k) for k in keys] + [("getMaxKey", None), ("getMinKey", None)] * 3000
for cls in [AllOneScan, AllOneLazyHeap, AllOne]:
    print(f"{cls.__name__:<15}: {bench(cls, worst):.3f} giây")

# Hạn chế bộ nhớ của cách 2: một key duy nhất được inc 5·10^4 lần
h2, b3 = AllOneLazyHeap(), AllOne()
for _ in range(5 * 10**4):
    h2.inc("a")
    b3.inc("a")
num_buckets, cur = 0, b3.head.next
while cur is not b3.tail:
    num_buckets += 1
    cur = cur.next
print("cách 2: số vé trong maxh =", len(h2.maxh), "| minh =", len(h2.minh))
print("cách 3: số bucket =", num_buckets, "| số key trong where =", len(b3.where))

AllOneScan     : 0.466 giây
AllOneLazyHeap : 0.002 giây
AllOne         : 0.001 giây
cách 2: số vé trong maxh = 50000 | minh = 50000
cách 3: số bucket = 1 | số key trong where = 1


## 11. Bảng so sánh các cách

`N` = số key đang tồn tại, `q` = số lời gọi.

| Cách | Ý tưởng | `inc` / `dec` | `getMaxKey` / `getMinKey` | Bộ nhớ | Nhận xét |
|---|---|---|---|---|---|
| 1. Dict + quét | đếm bằng dict, hỏi thì `max`/`min` | `O(1)` TB | **`O(N)`** | `O(N)` | dễ viết nhất, làm baseline để test |
| 2. Dict + 2 heap lazy | đẩy vé `(count, key)`, bỏ vé lỗi thời khi nổi lên đỉnh | `O(log q)` | `O(log q)` khấu hao | **`O(q)`** | kỹ thuật lazy deletion đáng học; không đạt `O(1)`, bộ nhớ theo lịch sử |
| 3. DLL bucket + dict | bucket theo count, xếp tăng dần, liên kết đôi, sentinel | **`O(1)`** TB | **`O(1)`** | `O(N)` | đáp ứng đúng yêu cầu đề; phải cẩn thận con trỏ |

TB = trung bình (do dùng hash table).

Bài học chuyển giao: khi giá trị chỉ thay đổi **từng nấc ±1**, hãy nghĩ tới **nhóm theo giá trị** và nối các nhóm thành **danh sách liên kết** — bước di chuyển luôn là sang "nhà hàng xóm", nên chèn / xóa đều `O(1)`.

## 12. Các lỗi người mới hay gặp

Các lỗi 1–4 dưới đây được viết thành class con của `AllOneBuckets`, mỗi class **chỉ** làm sai đúng một chỗ, để ta thấy hậu quả cụ thể.

### Lỗi 1: Trong `inc`, xóa bucket cũ **trước** rồi mới chèn bucket mới "sau nó"

```python
# SAI: dọn b trước
if b is not self.head:
    b.keys.remove(key)
    if not b.keys:
        self._remove(b)          # b đã bị gỡ khỏi danh sách...
nxt = b.next
if nxt is self.tail or nxt.count != c + 1:
    nxt = self._insert_after(b, c + 1)   # ...nhưng lại chèn "sau b"!
```

Vì sao sai: sau `_remove(b)`, các node hàng xóm **không còn trỏ tới `b`**, nhưng `b.prev` / `b.next` vẫn trỏ ra ngoài. Chèn "sau `b`" làm bucket mới chỉ được nối **một chiều**: `tail.prev` thấy nó nhưng `head.next` thì không. Hai chiều của danh sách mâu thuẫn nhau.

Sửa: tìm / tạo bucket `c + 1` **trước**, chuyển key, rồi mới dọn `b` (đúng thứ tự 3 bước ở mục 8.2).

In [24]:
class WrongIncOrder(AllOneBuckets):
    def inc(self, key):
        b = self.where.get(key, self.head)
        c = b.count
        if b is not self.head:
            b.keys.remove(key)
            if not b.keys:
                self._remove(b)                   # SAI: gỡ b trước
        nxt = b.next
        if nxt is self.tail or nxt.count != c + 1:
            nxt = self._insert_after(b, c + 1)    # chèn sau một node đã bị gỡ
        nxt.keys.add(key)
        self.where[key] = nxt


w = WrongIncOrder()
w.inc("a"); w.inc("a")
print("đi xuôi:", w.draw())                       # HEAD <-> TAIL  (mất bucket!)
print("getMaxKey:", repr(w.getMaxKey()), "| getMinKey:", repr(w.getMinKey()))   # 'a' và '' mâu thuẫn

đi xuôi: HEAD <-> TAIL
getMaxKey: 'a' | getMinKey: ''


### Lỗi 2: `dec` khi count là 1 nhưng không xử lý riêng → sinh "bucket count 0"

```python
# SAI: không có nhánh c == 1
prv = b.prev
if prv is self.head or prv.count != c - 1:
    prv = self._insert_after(b.prev, c - 1)   # c - 1 = 0 -> tạo bucket 0!
prv.keys.add(key)
```

Vì sao sai: đề nói count về 0 thì key **bị xóa khỏi cấu trúc**. Bucket 0 nằm ngay sau `head` nên `getMinKey` sẽ trả về một key "đã chết".

Sửa: nếu `c == 1` thì `del self.where[key]` và **không** thêm key vào bucket nào.

### Lỗi 3: Quên xóa bucket rỗng

Vì sao sai: invariant 2 bị phá. `head.next` có thể là bucket rỗng → `next(iter(set()))` ném `StopIteration`; hoặc đơn giản là bucket đầu không còn đại diện cho count nhỏ nhất.

Sửa: sau khi `b.keys.remove(key)`, luôn kiểm tra `if not b.keys: self._remove(b)`.

In [25]:
class WrongDecZero(AllOneBuckets):
    def dec(self, key):
        b = self.where[key]
        c = b.count
        prv = b.prev                                   # SAI: không có nhánh c == 1
        if prv is self.head or prv.count != c - 1:
            prv = self._insert_after(b.prev, c - 1)
        prv.keys.add(key)
        self.where[key] = prv
        b.keys.remove(key)
        if not b.keys:
            self._remove(b)


class WrongNoCleanup(AllOneBuckets):
    def inc(self, key):
        b = self.where.get(key, self.head)
        nxt = b.next
        if nxt is self.tail or nxt.count != b.count + 1:
            nxt = self._insert_after(b, b.count + 1)
        nxt.keys.add(key)
        self.where[key] = nxt
        if b is not self.head:
            b.keys.remove(key)                         # SAI: quên xóa b nếu rỗng


w2 = WrongDecZero()
for k in ["a", "b", "b"]:
    w2.inc(k)
w2.dec("a")
print("Lỗi 2:", w2.draw(), "| getMinKey =", repr(w2.getMinKey()), "(đúng phải là 'b')")

w3 = WrongNoCleanup()
w3.inc("a"); w3.inc("a")
print("Lỗi 3:", w3.draw())
try:
    w3.getMinKey()
except StopIteration:
    print("Lỗi 3: getMinKey -> StopIteration vì head.next là bucket rỗng")

Lỗi 2: HEAD <-> [0:{a}] <-> [2:{b}] <-> TAIL | getMinKey = 'a' (đúng phải là 'b')
Lỗi 3: HEAD <-> [1:{}] <-> [2:{a}] <-> TAIL
Lỗi 3: getMinKey -> StopIteration vì head.next là bucket rỗng


### Lỗi 4: Không kiểm tra count của bucket hàng xóm

Hai phiên bản của cùng một lỗi "lười kiểm tra":

- **Luôn dùng `b.next`** mà không xem `b.next.count == c + 1` chưa → key bị đưa nhầm lên count lớn hơn (ca #5 của trace: `b.next` là bucket 3 trong khi cần bucket 2).
- **Luôn tạo bucket mới** mà không xem `b.next` đã là bucket `c + 1` chưa → xuất hiện hai bucket cùng count, rồi thứ tự tăng dần bị phá, `getMinKey` / `getMaxKey` trả sai.

Sửa: đúng điều kiện `if nxt is self.tail or nxt.count != c + 1: tạo mới` (và đối xứng cho `dec`).

In [26]:
class WrongAlwaysReuse(AllOneBuckets):
    def inc(self, key):
        b = self.where.get(key, self.head)
        nxt = b.next
        if nxt is self.tail:                           # SAI: không so nxt.count với c + 1
            nxt = self._insert_after(b, b.count + 1)
        nxt.keys.add(key)
        self.where[key] = nxt
        if b is not self.head:
            b.keys.remove(key)
            if not b.keys:
                self._remove(b)


class WrongAlwaysNew(AllOneBuckets):
    def inc(self, key):
        b = self.where.get(key, self.head)
        nxt = self._insert_after(b, b.count + 1)       # SAI: luôn tạo bucket mới
        nxt.keys.add(key)
        self.where[key] = nxt
        if b is not self.head:
            b.keys.remove(key)
            if not b.keys:
                self._remove(b)


w4 = WrongAlwaysReuse()
for k in ["a", "b", "b", "b", "a"]:                    # đúng: a=2, b=3
    w4.inc(k)
print("luôn dùng b.next :", w4.draw(), "<- a bị ghi count 3, đúng phải là 2")

w5 = WrongAlwaysNew()
for k in ["a", "b", "b"]:                              # đúng: a=1, b=2
    w5.inc(k)
print("luôn tạo mới     :", w5.draw(), "| getMinKey =", repr(w5.getMinKey()), "(đúng phải là 'a')")

luôn dùng b.next : HEAD <-> [3:{a,b}] <-> TAIL <- a bị ghi count 3, đúng phải là 2
luôn tạo mới     : HEAD <-> [2:{b}] <-> [1:{a}] <-> TAIL | getMinKey = 'b' (đúng phải là 'a')


### Lỗi 5: Khai báo dict ở mức class thay vì trong `__init__`

```python
class AllOne:
    where = {}          # SAI: mọi object dùng CHUNG một dict này
```

Vì sao sai: LeetCode chạy nhiều test case trong cùng một tiến trình, mỗi test tạo một `AllOne()` mới. Thuộc tính ở mức class được **chia sẻ** giữa mọi object → dữ liệu của test trước "rò" sang test sau. Sửa: tạo `self.where = {}` (và `head`, `tail`) bên trong `__init__`.

In [27]:
class WrongShared:
    cnt = {}                                    # SAI: mức class

    def inc(self, key):
        self.cnt[key] = self.cnt.get(key, 0) + 1


x, y = WrongShared(), WrongShared()
x.inc("a")
print("object y chưa inc gì mà đã thấy:", y.cnt)    # {'a': 1}

object y chưa inc gì mà đã thấy: {'a': 1}


### Bảng tóm tắt lỗi

| # | Lỗi | Triệu chứng | Sửa |
|---:|---|---|---|
| 1 | Gỡ bucket cũ trước khi chèn bucket mới "sau nó" | hai chiều danh sách mâu thuẫn; `getMaxKey`/`getMinKey` trái ngược | chèn / chuyển key trước, dọn `b` sau |
| 2 | `dec` không xử lý riêng `count == 1` | xuất hiện bucket 0, `getMinKey` trả key đã bị xóa | `c == 1` → `del where[key]` |
| 3 | Quên xóa bucket rỗng | `StopIteration`, hoặc min/max sai | `if not b.keys: _remove(b)` |
| 4 | Không so `count` của bucket hàng xóm | key nhảy sai count / bucket trùng count, mất thứ tự | `nxt is tail or nxt.count != c + 1` |
| 5 | Thuộc tính ở mức class | test sau thấy dữ liệu test trước | khởi tạo trong `__init__` |
| 6 | Quên kiểm tra rỗng ở `getMaxKey`/`getMinKey` | `StopIteration` khi gọi trên sentinel; cách 1: `max()` báo `ValueError` | `if head.next is tail: return ""` |
| 7 | Chèn 4 con trỏ sai thứ tự (gán `p.next = new` trước) | mất dấu node đứng sau, danh sách bị cắt | gán `p.next = new` **sau cùng** |
| 8 | (Cách 2) đọc đỉnh heap mà không kiểm tra vé lỗi thời | trả key có count cũ hoặc key đã bị xóa | `while h and cnt.get(key) != count_on_ticket: heappop` |

## 13. Mẹo nhận dạng pattern, biến thể và bài liên quan

### Dấu hiệu nhận dạng

- Đề yêu cầu **mọi thao tác `O(1)`** và có hỏi **max / min / phần tử cũ nhất / phần tử ít dùng nhất** → gần như chắc chắn là **hash map + danh sách liên kết đôi**.
- Giá trị của một phần tử chỉ **tăng / giảm từng nấc 1** (tần suất, số lần truy cập) → nhóm theo giá trị thành **bucket**, nối các bucket theo thứ tự.
- Cần xóa một phần tử ở **giữa** khi đã cầm sẵn tham chiếu tới nó → liên kết **đôi** (có `prev`).
- Muốn bỏ hết các `if` biên đầu / cuối → **sentinel** `head` / `tail`.

### Template trong đầu

```text
dict: key -> node      (tìm node trong O(1))
DLL có sentinel:       (giữ thứ tự, chèn/xóa O(1))
  insert_after(p, new): new.prev=p; new.next=p.next; p.next.prev=new; p.next=new
  remove(x):            x.prev.next=x.next; x.next.prev=x.prev
min/max = head.next / tail.prev
```

### Biến thể

- Chỉ cần `getMaxKey` (không cần min) → `dict key → count` + `dict count → set` + biến `max_count` là đủ, vì khi bucket max rỗng thì max mới luôn là `max_count - 1` (xem LeetCode 895 ở bài tập 6).
- Thêm `getCount(key)` hoặc "trả **tất cả** key có count lớn nhất" → vẫn dùng cấu trúc bucket (bài tập 5).
- `inc` theo bước tùy ý `+d` → key không còn nhảy sang hàng xóm, phải tìm vị trí chèn → cần cây cân bằng / heap, mất tính `O(1)`.

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 146. LRU Cache (Medium) | dict + DLL có sentinel, đưa node vừa dùng về đầu |
| 460. LFU Cache (Hard) | "anh em ruột" của bài này: bucket theo tần suất, trong mỗi bucket giữ thứ tự LRU |
| 895. Maximum Frequency Stack (Hard) | gom phần tử theo tần suất, `max_freq` chỉ giảm từng nấc |
| 380. Insert Delete GetRandom O(1) (Medium) | thiết kế `O(1)` bằng dict + mảng |
| 716. Max Stack (Hard, Premium) | DLL + cấu trúc giữ max, xóa node ở giữa |

## 14. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 15) và **Đáp án** (mục 16, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

```text
["AllOne","inc","inc","inc","inc","inc","inc","dec","dec","getMaxKey","getMinKey",
 "dec","dec","getMaxKey","getMinKey","dec","dec","getMaxKey"]
[[],["x"],["y"],["y"],["z"],["z"],["z"],["z"],["z"],[],[],
 ["y"],["y"],[],[],["x"],["z"],[]]
```

Với mỗi lệnh `getMaxKey` / `getMinKey`, hãy ghi **tập các key có thể được trả về** (nhớ trường hợp hòa và trường hợp rỗng).

### Bài 2 — Vẽ tay danh sách bucket

Bắt đầu từ object rỗng, vẽ danh sách bucket (dạng `HEAD <-> [c:{...}] <-> ... <-> TAIL`) sau **mỗi** lệnh, và ghi rõ lệnh nào **tạo** / **xóa** bucket nào:

`inc a, inc b, inc a, inc a, inc c, dec a, dec b, inc c`

### Bài 3 — Điền TODO: method `inc`

Class dưới đây kế thừa `AllOneBuckets` (có sẵn `_insert_after`, `_remove`, `dec`, `getMaxKey`, `getMinKey`, `draw`). Bạn chỉ viết lại `inc` theo đúng 3 bước của mục 8.2.

In [28]:
class ExerciseInc(AllOneBuckets):
    def inc(self, key: str) -> None:
        # TODO 1: b = bucket hiện tại của key (key mới -> self.head)
        # TODO 2: nếu b.next là tail hoặc b.next.count != b.count + 1 -> tạo bucket mới ngay sau b
        # TODO 3: thêm key vào bucket count + 1, cập nhật self.where[key]
        # TODO 4: nếu b không phải head: bỏ key khỏi b.keys; b rỗng thì self._remove(b)
        pass


# Tự làm xong rồi mới mở test:
# got = run_commands(ExerciseInc, EX1_OPS, EX1_ARGS)
# assert got == EX1_OUT
# for _ in range(200):
#     cross_check(random_commands(60, list("abcd")), [ExerciseInc])

### Bài 4 — Tự viết lại toàn bộ từ trang trắng

Không nhìn lên trên, tự viết class `MyAllOne` gồm: class node, hai sentinel, dict `where`, `_insert_after`, `_remove`, `inc`, `dec`, `getMaxKey`, `getMinKey`. Mục tiêu: viết đúng ngay lần đầu các phép gán con trỏ.

In [29]:
class MyAllOne:
    def __init__(self):
        # TODO: tạo head, tail (nối với nhau) và self.where = {}
        raise NotImplementedError

    def inc(self, key: str) -> None:
        raise NotImplementedError

    def dec(self, key: str) -> None:
        raise NotImplementedError

    def getMaxKey(self) -> str:
        raise NotImplementedError

    def getMinKey(self) -> str:
        raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert run_commands(MyAllOne, EX1_OPS, EX1_ARGS) == EX1_OUT
# for _ in range(300):
#     cross_check(random_commands(80, list("abcde"), 0.4, 0.4), [MyAllOne])

### Bài 5 — Biến thể: hỏi count và lấy **tất cả** key max / min

Mở rộng `AllOneBuckets` thêm 3 method:

- `getCount(key)`: count hiện tại của `key` (chưa có thì `0`) — `O(1)`.
- `getAllMaxKeys()`: list **đã sắp xếp** mọi key có count lớn nhất (rỗng thì `[]`).
- `getAllMinKeys()`: tương tự cho count nhỏ nhất.

Câu hỏi thêm: `getAllMaxKeys` tốn bao nhiêu thời gian? Vì sao không thể là `O(1)`?

In [30]:
class AllOneExtra(AllOneBuckets):
    def getCount(self, key: str) -> int:
        # TODO: dùng self.where
        pass

    def getAllMaxKeys(self) -> list:
        # TODO: bucket cuối = self.tail.prev (nhớ trường hợp rỗng)
        pass

    def getAllMinKeys(self) -> list:
        pass


# Tự làm xong rồi mới mở test:
# e = AllOneExtra()
# for k in ["a", "b", "b", "c", "c"]:
#     e.inc(k)
# assert e.getCount("b") == 2 and e.getCount("zzz") == 0
# assert e.getAllMaxKeys() == ["b", "c"] and e.getAllMinKeys() == ["a"]

### Bài 6 — Mini challenge: LeetCode 895. Maximum Frequency Stack

Thiết kế `FreqStack`:

- `push(val)`: đẩy số nguyên `val` lên đỉnh stack.
- `pop()`: xóa và trả về phần tử **xuất hiện nhiều nhất** trong stack. Nếu hòa, chọn phần tử **gần đỉnh nhất** (được đẩy gần đây nhất).

Ví dụ đề:

```text
["FreqStack","push","push","push","push","push","push","pop","pop","pop","pop"]
[[],[5],[7],[5],[7],[4],[5],[],[],[],[]]
Output: [null,null,null,null,null,null,null,5,7,5,4]
```

Yêu cầu: mọi thao tác `O(1)` trung bình. Gợi ý liên hệ: cũng là "gom phần tử theo tần suất" như bucket của bài 432, nhưng ở đây tần suất max chỉ giảm **từng nấc**, nên không cần danh sách liên kết.

In [31]:
class FreqStack:
    def __init__(self):
        # TODO: freq (val -> tần suất), group (tần suất -> list các val theo thứ tự đẩy), max_freq
        pass

    def push(self, val: int) -> None:
        pass

    def pop(self) -> int:
        pass


# Tự làm xong rồi mới mở test:
# ops = ["FreqStack","push","push","push","push","push","push","pop","pop","pop","pop"]
# args = [[],[5],[7],[5],[7],[4],[5],[],[],[],[]]
# assert run_commands(FreqStack, ops, args) == [None]*7 + [5, 7, 5, 4]

## 15. Gợi ý theo mức

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Lập bảng count sau mỗi lệnh như mục 1.1. | Hòa thì mọi key cùng count max/min đều hợp lệ; không còn key → `""`. | Sau 8 lệnh đầu: `x:1, y:2, z:1`. |
| 2 | Mỗi lệnh chỉ đổi count của **một** key, sang bucket kề bên. | `inc`: `b.next` có đúng count `c+1` không? `dec`: `b.prev` có đúng `c-1` không? Bucket cũ còn key không? | Sau `inc a` lần 3: `[1:{b}] [3:{a}]`. |
| 3 | 3 bước: tìm/tạo → chuyển → dọn. | `b = self.where.get(key, self.head)`; điều kiện tạo: `nxt is self.tail or nxt.count != b.count + 1`. | `nxt = self._insert_after(b, b.count + 1)`; cuối cùng `if b is not self.head: b.keys.remove(key); if not b.keys: self._remove(b)`. |
| 4 | Vẽ hình 4 con trỏ trước khi viết `_insert_after`. | Sentinel `head.next = tail`, `tail.prev = head`; `dec` có nhánh `count == 1`. | Chèn trước `b` = `insert_after(b.prev, ...)`. |
| 5 | Mọi key max nằm chung **một** bucket. | `self.tail.prev is self.head` nghĩa là rỗng. | `sorted(self.tail.prev.keys)`; thời gian `O(k log k)` với `k` = số key được trả. |
| 6 | Mỗi lần push val với tần suất mới `f`, đẩy val vào "stack của tần suất `f`". | `group[f]` là list dùng như stack; `max_freq` = tần suất lớn nhất. | pop: `val = group[max_freq].pop()`; `freq[val] -= 1`; nếu `group[max_freq]` rỗng thì `max_freq -= 1`. |

## 16. Đáp án tham khảo

### Đáp án Bài 1

| Sau lệnh | Count | Lệnh hỏi | Các key hợp lệ |
|---|---|---|---|
| `inc x, inc y ×2, inc z ×3, dec z ×2` | `x:1, y:2, z:1` | `getMaxKey` | `{"y"}` |
| | | `getMinKey` | `{"x", "z"}` (hòa count 1) |
| `dec y ×2` | `x:1, z:1` (y bị xóa) | `getMaxKey` | `{"x", "z"}` |
| | | `getMinKey` | `{"x", "z"}` |
| `dec x, dec z` | rỗng | `getMaxKey` | `{""}` |

Cell dưới chạy thật bằng `AllOne` và kiểm tra mỗi output nằm trong tập dự đoán.

In [32]:
B1_OPS = ["AllOne", "inc", "inc", "inc", "inc", "inc", "inc", "dec", "dec", "getMaxKey", "getMinKey",
          "dec", "dec", "getMaxKey", "getMinKey", "dec", "dec", "getMaxKey"]
B1_ARGS = [[], ["x"], ["y"], ["y"], ["z"], ["z"], ["z"], ["z"], ["z"], [], [],
           ["y"], ["y"], [], [], ["x"], ["z"], []]
got = run_commands(AllOne, B1_OPS, B1_ARGS)
answers = [{"y"}, {"x", "z"}, {"x", "z"}, {"x", "z"}, {""}]
asked = [out for op, out in zip(B1_OPS, got) if op.startswith("get")]
for out, ok in zip(asked, answers):
    assert out in ok
print("output thật:", show(got))
print("các lệnh hỏi:", asked, "-> đều nằm trong tập dự đoán")

output thật: [null, null, null, null, null, null, null, null, null, "y", "x", null, null, "x", "x", null, null, ""]
các lệnh hỏi: ['y', 'x', 'x', 'x', ''] -> đều nằm trong tập dự đoán


### Đáp án Bài 2

| Lệnh | Tạo / xóa | Danh sách |
|---|---|---|
| `inc a` | tạo 1 | `[1:{a}]` |
| `inc b` | — | `[1:{a,b}]` |
| `inc a` | tạo 2 | `[1:{b}] [2:{a}]` |
| `inc a` | tạo 3, xóa 2 | `[1:{b}] [3:{a}]` |
| `inc c` | — | `[1:{b,c}] [3:{a}]` |
| `dec a` | tạo 2 (giữa 1 và 3), xóa 3 | `[1:{b,c}] [2:{a}]` |
| `dec b` | — (`b` về 0, bucket 1 vẫn còn `c`) | `[1:{c}] [2:{a}]` |
| `inc c` | xóa 1 (gộp vào 2) | `[2:{a,c}]` |

In [33]:
trace_buckets(AllOneBuckets(), [("inc", "a"), ("inc", "b"), ("inc", "a"), ("inc", "a"),
                                ("inc", "c"), ("dec", "a"), ("dec", "b"), ("inc", "c")])

#1  inc('a'): a 0->1 | tạo 1
     HEAD <-> [1:{a}] <-> TAIL
#2  inc('b'): b 0->1 | không tạo/xóa
     HEAD <-> [1:{a,b}] <-> TAIL
#3  inc('a'): a 1->2 | tạo 2
     HEAD <-> [1:{b}] <-> [2:{a}] <-> TAIL
#4  inc('a'): a 2->3 | tạo 3, xóa 2
     HEAD <-> [1:{b}] <-> [3:{a}] <-> TAIL
#5  inc('c'): c 0->1 | không tạo/xóa
     HEAD <-> [1:{b,c}] <-> [3:{a}] <-> TAIL
#6  dec('a'): a 3->2 | tạo 2, xóa 3
     HEAD <-> [1:{b,c}] <-> [2:{a}] <-> TAIL
#7  dec('b'): b 1->0 | không tạo/xóa
     HEAD <-> [1:{c}] <-> [2:{a}] <-> TAIL
#8  inc('c'): c 1->2 | xóa 1
     HEAD <-> [2:{a,c}] <-> TAIL


### Đáp án Bài 3 và Bài 4

Bài 3 là đúng method `inc` của mục 8.2. Bài 4 là một bản viết lại hoàn chỉnh, dùng dict để làm node (cho thấy node không nhất thiết phải là class riêng — chỉ cần có đủ 4 "ô": `count`, `keys`, `prev`, `next`). Cả hai được test chéo với cách 1.

In [34]:
class ExerciseIncAnswer(AllOneBuckets):
    def inc(self, key: str) -> None:
        b = self.where.get(key, self.head)
        nxt = b.next
        if nxt is self.tail or nxt.count != b.count + 1:
            nxt = self._insert_after(b, b.count + 1)
        nxt.keys.add(key)
        self.where[key] = nxt
        if b is not self.head:
            b.keys.remove(key)
            if not b.keys:
                self._remove(b)


class MyAllOneAnswer:
    def __init__(self):
        self.head = {"count": 0, "keys": set(), "prev": None, "next": None}
        self.tail = {"count": 0, "keys": set(), "prev": None, "next": None}
        self.head["next"], self.tail["prev"] = self.tail, self.head
        self.where = {}

    def _insert_after(self, p, count):
        new = {"count": count, "keys": set(), "prev": p, "next": p["next"]}
        p["next"]["prev"] = new
        p["next"] = new
        return new

    def _remove(self, x):
        x["prev"]["next"] = x["next"]
        x["next"]["prev"] = x["prev"]

    def inc(self, key: str) -> None:
        b = self.where.get(key, self.head)
        nxt = b["next"]
        if nxt is self.tail or nxt["count"] != b["count"] + 1:
            nxt = self._insert_after(b, b["count"] + 1)
        nxt["keys"].add(key)
        self.where[key] = nxt
        if b is not self.head:
            b["keys"].remove(key)
            if not b["keys"]:
                self._remove(b)

    def dec(self, key: str) -> None:
        b = self.where[key]
        if b["count"] == 1:
            del self.where[key]
        else:
            prv = b["prev"]
            if prv is self.head or prv["count"] != b["count"] - 1:
                prv = self._insert_after(b["prev"], b["count"] - 1)
            prv["keys"].add(key)
            self.where[key] = prv
        b["keys"].remove(key)
        if not b["keys"]:
            self._remove(b)

    def getMaxKey(self) -> str:
        return "" if self.tail["prev"] is self.head else next(iter(self.tail["prev"]["keys"]))

    def getMinKey(self) -> str:
        return "" if self.head["next"] is self.tail else next(iter(self.head["next"]["keys"]))


for cls in [ExerciseIncAnswer, MyAllOneAnswer]:
    assert run_commands(cls, EX1_OPS, EX1_ARGS) == EX1_OUT
random.seed(43)
for _ in range(300):
    cross_check(random_commands(80, list("abcde"), 0.4, 0.4), [ExerciseIncAnswer, MyAllOneAnswer])
print("Bài 3 + Bài 4: Example 1 đúng, 300 chuỗi lệnh ngẫu nhiên khớp cách 1")

Bài 3 + Bài 4: Example 1 đúng, 300 chuỗi lệnh ngẫu nhiên khớp cách 1


### Đáp án Bài 5

- `getCount`: `O(1)` trung bình — chỉ tra `where`.
- `getAllMaxKeys` / `getAllMinKeys`: phải **liệt kê** `k` key trong bucket nên tối thiểu `O(k)`; thêm sắp xếp thành `O(k log k)`. Không thể `O(1)` vì riêng việc tạo list kết quả dài `k` đã tốn `k` bước.

In [35]:
class AllOneExtraAnswer(AllOneBuckets):
    def getCount(self, key: str) -> int:
        return self.where[key].count if key in self.where else 0

    def getAllMaxKeys(self) -> list:
        return [] if self.tail.prev is self.head else sorted(self.tail.prev.keys)

    def getAllMinKeys(self) -> list:
        return [] if self.head.next is self.tail else sorted(self.head.next.keys)


e = AllOneExtraAnswer()
assert e.getAllMaxKeys() == [] and e.getCount("a") == 0
for k in ["a", "b", "b", "c", "c"]:
    e.inc(k)
assert e.getCount("b") == 2 and e.getCount("zzz") == 0
assert e.getAllMaxKeys() == ["b", "c"] and e.getAllMinKeys() == ["a"]
e.dec("a")
assert e.getAllMinKeys() == ["b", "c"] and e.getCount("a") == 0
print(e.draw(), "| max:", e.getAllMaxKeys(), "| min:", e.getAllMinKeys())

HEAD <-> [2:{b,c}] <-> TAIL | max: ['b', 'c'] | min: ['b', 'c']


### Đáp án Bài 6 — FreqStack

Ý tưởng: `group[f]` là một stack chứa các giá trị **lúc chúng đạt tần suất `f`**. Một giá trị xuất hiện 3 lần sẽ có mặt trong `group[1]`, `group[2]`, `group[3]`. Khi `pop`, lấy đỉnh của `group[max_freq]` — vừa là phần tử tần suất lớn nhất, vừa là cái được đẩy gần nhất trong số đó. Nếu `group[max_freq]` rỗng thì `max_freq` giảm đúng 1 — vì mọi giá trị có tần suất `max_freq` cũng đã có mặt trong `group[max_freq - 1]`. Đây chính là quan sát "chỉ nhảy sang hàng xóm" của bài 432, nên không cần danh sách liên kết.

In [36]:
class FreqStackAnswer:
    def __init__(self):
        self.freq = {}          # val -> tần suất hiện tại
        self.group = {}         # tần suất -> list (stack) các val
        self.max_freq = 0

    def push(self, val: int) -> None:
        f = self.freq.get(val, 0) + 1
        self.freq[val] = f
        if f not in self.group:
            self.group[f] = []
        self.group[f].append(val)
        self.max_freq = max(self.max_freq, f)

    def pop(self) -> int:
        val = self.group[self.max_freq].pop()
        self.freq[val] -= 1
        if not self.group[self.max_freq]:
            self.max_freq -= 1
        return val


ops = ["FreqStack", "push", "push", "push", "push", "push", "push", "pop", "pop", "pop", "pop"]
args = [[], [5], [7], [5], [7], [4], [5], [], [], [], []]
got = run_commands(FreqStackAnswer, ops, args)
print(show(got))
assert got == [None] * 7 + [5, 7, 5, 4]


def freq_stack_brute(pushes_pops):
    # brute force: list thô, mỗi lần pop quét tìm phần tử tần suất max gần đỉnh nhất
    st, out = [], []
    for x in pushes_pops:
        if x is None:
            cnt = {}
            for v in st:
                cnt[v] = cnt.get(v, 0) + 1
            best = max(cnt.values())
            for i in range(len(st) - 1, -1, -1):
                if cnt[st[i]] == best:
                    out.append(st.pop(i))
                    break
        else:
            st.append(x)
    return out


random.seed(895)
for _ in range(300):
    seq, size = [], 0
    for _ in range(random.randint(1, 40)):
        if size and random.random() < 0.4:
            seq.append(None); size -= 1
        else:
            seq.append(random.randint(1, 4)); size += 1
    fs, res = FreqStackAnswer(), []
    for x in seq:
        if x is None:
            res.append(fs.pop())
        else:
            fs.push(x)
    assert res == freq_stack_brute(seq)
print("FreqStack khớp brute force trên 300 chuỗi ngẫu nhiên")

[null, null, null, null, null, null, null, 5, 7, 5, 4]


FreqStack khớp brute force trên 300 chuỗi ngẫu nhiên


## 17. Template cuối cùng nên nhớ

```python
class Node:
    def __init__(self, count):
        self.count, self.keys, self.prev, self.next = count, set(), None, None

class AllOne:
    def __init__(self):
        self.head, self.tail = Node(0), Node(0)
        self.head.next, self.tail.prev = self.tail, self.head
        self.where = {}

    def _insert_after(self, p, count):
        new = Node(count)
        new.prev, new.next = p, p.next
        p.next.prev = new
        p.next = new                      # sau cùng
        return new

    def _remove(self, x):
        x.prev.next, x.next.prev = x.next, x.prev

    def inc(self, key):
        b = self.where.get(key, self.head)                   # key mới: head = "count 0"
        nxt = b.next
        if nxt is self.tail or nxt.count != b.count + 1:
            nxt = self._insert_after(b, b.count + 1)         # 1. tìm/tạo
        nxt.keys.add(key); self.where[key] = nxt             # 2. chuyển
        if b is not self.head:                               # 3. dọn
            b.keys.remove(key)
            if not b.keys: self._remove(b)

    def dec(self, key):
        b = self.where[key]
        if b.count == 1:
            del self.where[key]                              # về 0: biến mất
        else:
            prv = b.prev
            if prv is self.head or prv.count != b.count - 1:
                prv = self._insert_after(b.prev, b.count - 1)
            prv.keys.add(key); self.where[key] = prv
        b.keys.remove(key)
        if not b.keys: self._remove(b)

    def getMaxKey(self):
        return "" if self.tail.prev is self.head else next(iter(self.tail.prev.keys))

    def getMinKey(self):
        return "" if self.head.next is self.tail else next(iter(self.head.next.keys))
```

**Câu thần chú:** *"Gom key theo count thành bucket; bucket xếp tăng dần trong danh sách liên kết đôi có sentinel; key chỉ nhảy sang bucket hàng xóm: tìm/tạo → chuyển → dọn. Min ở `head.next`, max ở `tail.prev`."*

## 18. Checklist tự kiểm tra trước khi submit

- Tên class và method đúng y như đề: `AllOne`, `inc`, `dec`, `getMaxKey`, `getMinKey`; trả `""` (không phải `None`) khi rỗng?
- Mọi trạng thái (`head`, `tail`, `where`) được tạo **trong `__init__`**, và `head.next = tail`, `tail.prev = head`?
- `_insert_after` gán đủ **4** con trỏ, `p.next = new` làm sau cùng? `_remove` gán đủ **2** con trỏ?
- `inc`: key mới dùng `head` làm "bucket count 0"; có kiểm tra `nxt is tail or nxt.count != c + 1` trước khi tạo bucket?
- `inc`: chèn / chuyển key **trước**, dọn bucket cũ **sau**; không `remove` key khỏi `head`?
- `dec`: có nhánh riêng `count == 1` (xóa khỏi `where`, không tạo bucket 0)? Chèn trước `b` bằng `insert_after(b.prev, ...)`?
- Sau khi bỏ key khỏi bucket cũ, luôn xóa bucket nếu rỗng?
- `getMaxKey` / `getMinKey` kiểm tra rỗng bằng sentinel trước khi `next(iter(...))`?
- Đã test: Example 1, hỏi khi rỗng, `dec` về 0, nhiều key hòa (kiểm theo **count**), key quay lại sau khi bị xóa, chuỗi lên cao rồi xuống (tạo bucket ở giữa), và test chéo ngẫu nhiên với bản quét `O(N)`?